# Procedurally Optimised ZX-Diagram Cutting

##### This Jupyter notebook is supplementary to the paper on Procedurally Optimised ZX-Diagram Cutting (2024), by Matthew Sutcliffe and Aleks Kissinger

## Initialisation

In [1]:
#pip install pyzx

In [2]:
import pyzx as zx

import sys, os, math
import random
import sympy as sym
from fractions import Fraction
from pyzx import print_matrix
from pyzx.basicrules import *
from pyzx.graph.graph import GraphS
from pyzx.graph.base import BaseGraph

## Introduction with example circuits

## Example 1

##### Let's generate an example circuit and partially simplify it, while keeping its structure graph-like...

In [3]:
strCirc = """
qreg q[8];
rz(0.25*pi) q[7];
cx q[5], q[7];
cx q[1], q[5];
cx q[4], q[5];
rz(0.25*pi) q[7];
cx q[6], q[7];
rz(0.25*pi) q[7];
cx q[5], q[7];
cx q[4], q[5];
rz(0.25*pi) q[7];
cx q[3], q[7];
rz(0.25*pi) q[7];
cx q[5], q[7];
cx q[4], q[5];
rz(0.25*pi) q[7];
cx q[2], q[7];
rz(0.25*pi) q[7];
cx q[5], q[7];
rz(0.25*pi) q[7];
"""

c = zx.qasm(strCirc)
g = c.to_graph()
#g.normalize()
zx.draw(g, labels=True,scale=30)
print("T-count = ", zx.tcount(g))

T-count =  8


In [4]:
zx.basicrules.fuse(g,22,14)
zx.basicrules.fuse(g,22,30)

zx.basicrules.fuse(g,11,13)

# This just re-numbers the vertex indexes to remove any gaps:
h = g.copy()
g = h.copy()

zx.draw(g, labels=True,scale=30)
print("T-count = ", zx.tcount(g))

T-count =  8


##### As outlined in the paper, we can define a procedure to compute the weights of the vertices (at the base tier)...

In [5]:
def compFirstTier(g: GraphS,showWeights=True):
    vweights = [0]*(max(g.vertices())+1)
    vDoorsteps = [set() for i in range(max(g.vertices())+1)] # Doorstep nodes are T-gates on the doorstep to be cancelled if the block is removed
    vCtrls = [set() for i in range(max(g.vertices())+1)] # This keeps track of the map for all red nodes of a CNOT -> their set of control nodes


    # vweights = dict()
    # vDoorsteps = dict() # Doorstep nodes are T-gates on the doorstep to be cancelled if the block is removed
    # vCtrls = dict() # This keeps track of the map for all red nodes of a CNOT -> their set of control nodes


    for v in g.vertices(): # Be careful as g.vertices isn't normalised
        vweights[v] = 0
        # vDoorsteps[v] = set()
        # vCtrls[v] = set()
        #if g.phase(v) in (0.25, 0.75, 1.25, 1.75): vweights[v] = 1 # If the vertex is T-like, give it a base weighting of 1

    for v in g.vertices():
        if g.type(v) == 2 and g.phase(v) in {0,1}: # If red vertex of n*pi-phase
            for neigh in g.neighbors(v):
                if g.qubit(neigh) != g.qubit(v): # If the neighbour is not on the same qubit (hence is a control)
                    vCtrls[v].add(neigh) # Keep track of the CNOT's control spiders

    for v in g.vertices():
        if g.type(v) == 2 and g.phase(v) in {0,1}: # If red vertex of n*pi-phase
            Tneighs = 0
            countedNeighs = set() # Keeps track of which T-gates have already been counted for this batch, to avoid overcounting
            for neigh in g.neighbors(v):
                if g.qubit(neigh) == g.qubit(v): # If the neighbour is on the same qubit
                    if g.phase(neigh) in (0.25, 0.75, 1.25, 1.75): # If the neighbour also is T-like
                        if not(neigh in countedNeighs): # If the neighbour hasn't already been counted yet
                            Tneighs += 1
                            countedNeighs.add(neigh)

            #print()
            #print(v, "\t", Tneighs)
            #print(countedNeighs)

            if Tneighs == 2 and len(vCtrls[v])>0: # the second check is a temporary fix to an occasional bug
                #if (len(vCtrls[v])==0): print("\n\nERROR at vertex",v); zx.draw(g,labels=True,scale=30); #TEMP
                w = 2/(len(vCtrls[v])) # If this gives div-by-0 error, then partialSimp() is probably missing somewhere
                for ctrl in vCtrls[v]:

                    isNew = True
                    for i in countedNeighs:
                        if (i in vDoorsteps[ctrl]): isNew = False

                    if isNew:
                        for i in countedNeighs: vDoorsteps[ctrl].add(i)
                        vweights[ctrl] += w
                        
    #-----
    
    tier = 1
    vtiers = newvweights = [0]*(max(g.vertices())+1)
    # vtiers = newvweights = {v: 0 for v in g.vertices()}

    # Weightings for lowest tier:
    if (showWeights): print("--== TIER 1 ==--\nvertex \t weight\t children")
    for i in range(len(vweights)):
    # for i, w in vweights.items():
        # if not vDoorsteps[i]: continue # skip if empty
        if len(vDoorsteps[i]) <= 0: continue # skip if empty
        # if (len(vDoorsteps[i]) > 0):
        w = vweights[i]
        #if (g.phase(i) in [0.25,0.75,1.25,1.75]): w += 1 # +1 if the vertex itself is T-like (not seen by others)
        if (showWeights): print(i, "\t", w, "\t", vDoorsteps[i])
        vtiers[i] = tier
    if (showWeights): print()
    
    vweights_0 = vweights.copy()
    vDoorsteps_0 = vDoorsteps.copy()

    vweights_max = vweights.copy()
    #vDoorsteps_max = vDoorsteps.copy()
    
    return [tier,vweights_max,vweights,vtiers,vDoorsteps]
    
tierData     = compFirstTier(g)
tier         = tierData[0]
vweights_max = tierData[1]
vweights     = tierData[2]
vtiers       = tierData[3]
vDoorsteps   = tierData[4]

--== TIER 1 ==--
vertex 	 weight	 children
10 	 2.0 	 {8, 13}
15 	 2.0 	 {16, 13}
18 	 2.0 	 {16, 21}
23 	 2.0 	 {24, 21}
26 	 2.0 	 {24, 28}
30 	 2.0 	 {28, 31}
33 	 2.0 	 {34, 31}



##### Here, we observe 7 vertices with weights of 2.0. These are the vertices which, if cut, would enable 2 T-like gates to reduce to Clifford (hence reducing the T-count by 2). (We note also which T-like vertices they are preventing from reducing in this way.)

##### When looking through the scope of the NEXT tier, we're considering cuts as prerequesites to those identified in the base tier. In other words, we're considering vertices which, rather than blocking T-like children from fusing, are blocking tier-1 WEIGHTED vertices from fusing. So, we're looking for vertices which, when cut (along with its subsequently fused children then also being cut), are ultimately likely to be worthwhile in reducing T-count efficiently. And, now that we're requiring a cut on a vertex AND on its to-be fused children, the existing weights AS SEEN BY THE NEXT TIER are halved (though capped to 1.0, meaning an almost certain worthwhile cut). So, the above weightings as seen by the next tier are...

In [6]:
# Weightings as seen by next tier:
for i in range(len(vweights)):
    if (len(vDoorsteps[i]) > 0):
        w = min(vweights[i]/2,1.0)
        print(i, "\t", w, "\t", vDoorsteps[i])

10 	 1.0 	 {8, 13}
15 	 1.0 	 {16, 13}
18 	 1.0 	 {16, 21}
23 	 1.0 	 {24, 21}
26 	 1.0 	 {24, 28}
30 	 1.0 	 {28, 31}
33 	 1.0 	 {34, 31}


##### So now let's compute the next tier, similar to the way we did the first (excpet looking for fusing opportunities of weighted vertices rather than T-like vertices)...

In [7]:
def compNextTier(g: GraphS,showWeights: bool,tier: int,vweights_max: list[int], vweights: list[int], 
                 vtiers: list[int]):
    isEmptyTier = False
    while (isEmptyTier == False):
        tier += 1
        newvweights = [0]*(max(g.vertices())+1)
        vDoorsteps = [set() for i in range(max(g.vertices())+1)] # Doorstep nodes are weighted nodes on the doorstep to be cancelled if the block is removed
        vCtrls = [set() for i in range(max(g.vertices())+1)] # This keeps track of the map for all red nodes of a CNOT -> their set of control nodes

        # newvweights = dict()
        # vDoorsteps = dict() # Doorstep nodes are weighted nodes on the doorstep to be cancelled if the block is removed
        # vCtrls = dict() # This keeps track of the map for all red nodes of a CNOT -> their set of control nodes
        # print(g.vertices())
        for v in g.vertices(): # Be careful as g.vertices isn't normalised
            vweights[v] = min(vweights_max[v]/2,1.0) # Update lower tiers' values for higher tier perspective
            newvweights[v] = 0
            # vDoorsteps[v] = set()
            # vCtrls[v] = set()
            #if g.phase(v) in (0.25, 0.75, 1.25, 1.75): vweights[v] = 1 # If the vertex is T-like, give it a base weighting of 1

        for v in g.vertices():
            if g.type(v) == 2 and g.phase(v) in {0,1}: # If red vertex of n*pi-phase
                for neigh in g.neighbors(v):
                    if g.qubit(neigh) != g.qubit(v): # If the neighbour is not on the same qubit (hence is a control)
                        vCtrls[v].add(neigh) # Keep track of the CNOT's control spiders

        for v in g.vertices():
            if g.type(v) == 2 and g.phase(v) in {0,1}: # If red vertex of n*pi-phase
                Tneighs = 0
                countedNeighs = set() # Keeps track of which weighted nodes have already been counted for this batch, to avoid overcounting
                for neigh in g.neighbors(v):
                    if g.qubit(neigh) == g.qubit(v): # If the neighbour is on the same qubit
                        if vweights[neigh] > 0: # If the neighbour also is weighted
                            if not(neigh in countedNeighs): # If the neighbour hasn't already been counted yet
                                Tneighs += 1
                                countedNeighs.add(neigh)

                #print()
                #print(v, "\t", Tneighs)
                #print(countedNeighs)

                if Tneighs == 2 and len(vCtrls[v])>0: # the second check is a temporary fix to an occasional bug
                    #if (len(vCtrls[v])==0): print("\n\nERROR at vertex",v); zx.draw(g,labels=True,scale=30); #TEMP
                    w = 2/(len(vCtrls[v])) # If this gives div-by-0 error, then partialSimp() is probably missing somewhere
                    for ctrl in vCtrls[v]:
                        isNew = True

                        for i in countedNeighs: # avoid double-counting
                            if (i in vDoorsteps[ctrl]): isNew = False

                        isPrevTier = False
                        for i in countedNeighs: # at least one must be of the immediately previous tier
                            if (vtiers[i] == tier-1): isPrevTier = True
                        if (isPrevTier==False): isNew = False

                        if isNew:
                            for i in countedNeighs: vDoorsteps[ctrl].add(i)
                            newvweights[ctrl] += w


        #-----
        
        if (showWeights): print("--== TIER",tier,"==--")

        isEmptyTier = True
        for i in range(len(vweights)):
        # for i in vweights.keys():
            vweights[i] = newvweights[i] # Update vweights from buffer

            if (vweights[i] > vweights_max[i]): vweights_max[i] = vweights[i] # Update max vals

            if (vweights[i]>0): #if (len(vDoorsteps[i]) > 0):
                if (showWeights): print(i, "\t", vweights[i], "\t", vDoorsteps[i])
                vtiers[i] = tier
                isEmptyTier = False

        if (showWeights):
            if (isEmptyTier): print("(empty)")
            print()
            return [tier,vweights_max,vweights,vtiers,vDoorsteps,False] # If no more tiers, return False in [5]

        return [tier,vweights_max,vweights,vtiers,vDoorsteps,True]
    
tierData     = compNextTier(g,True,tier,vweights_max,vweights,vtiers)
tier         = tierData[0]
vweights_max = tierData[1]
vweights     = tierData[2]
vtiers       = tierData[3]
vDoorsteps   = tierData[4]

--== TIER 2 ==--
12 	 1.0 	 {10, 18}
20 	 3.0 	 {26, 33, 10, 18}



##### If we were to check the NEXT tier (tier 3), we find - in this case - no tier-3 vertex weightings, and so we stop here...

In [8]:
compNextTier(g,True,tier,vweights_max,vweights,vtiers);

--== TIER 3 ==--
(empty)



##### Looking at the total data we have so far, we see...

In [9]:
# Total weightings...

def dispWeights(tier,vweights_max,vweights,vtiers):
    print("vertex\t weight\t\t tier")

    for i in range(len(vweights)):
        if (vweights_max[i]>0): #if (len(vDoorsteps[i]) > 0):
            w = vweights_max[i]
            b = 0
            if (g.phase(i) in [0.25,0.75,1.25,1.75]): b+=1 # Add 1 to weight if the prospective vertex to cut is itself T-like
            strB = "(+" + str(b) + ")" #strB = " (" + str(w+b) + ")"
            print(i, "\t", w,strB, "\t", vtiers[i])
            
dispWeights(tier,vweights_max,vweights,vtiers)

vertex	 weight		 tier
10 	 2.0 (+0) 	 1
12 	 1.0 (+0) 	 2
15 	 2.0 (+0) 	 1
18 	 2.0 (+0) 	 1
20 	 3.0 (+0) 	 2
23 	 2.0 (+0) 	 1
26 	 2.0 (+0) 	 1
30 	 2.0 (+0) 	 1
33 	 2.0 (+0) 	 1


##### Here, the bonus (+0)/(+1) on the weights are determined by whether that vertex is itself T-like. This is not seen by other vertices when considering the weights of their neighbours (as fusing a couple of tiered-vertices where one is T-like has no extra benefit than if neither were T-like), but is only seen when considering which weight is best to cut first (as cutting a T-like tiered-vertex will remove 1 extra T-gate than if it had not been T-like).

##### Higher tier vertices are always considered preferentially to lower tier vertices when deciding which to cut next. So, in this example, the initial best cut is on vertex 20, as it is on tier 2 and has the max weight there of 3.0 (even though this cut itself does not reduce the T-count at all).

##### Before we proceed to the next step, let's look at another (more complex) example, and see if you can work through the how its results are arrived at...

## Example 2

In [10]:
strCirc = """
qreg q[8];
rz(1.25*pi) q[4];
rz(0.75*pi) q[5];

rz(0.25*pi) q[7];
cx q[5], q[7];

cx q[1], q[5];

cx q[4], q[5];
rz(0.75*pi) q[5];

rz(0.25*pi) q[7];
cx q[6], q[7];
rz(0.25*pi) q[7];
cx q[5], q[7];

cx q[4], q[5];
rz(0.75*pi) q[5];

rz(0.25*pi) q[7];
cx q[3], q[7];
rz(0.25*pi) q[7];
cx q[5], q[7];

cx q[4], q[5];
rz(0.75*pi) q[5];

rz(0.25*pi) q[7];
cx q[2], q[7];
rz(0.25*pi) q[7];
cx q[5], q[7];
rz(0.25*pi) q[7];
"""

c = zx.qasm(strCirc)
g = c.to_graph()
#g.normalize()
zx.draw(g, labels=True,scale=30)
print("T-count = ", zx.tcount(g))

T-count =  13


In [11]:
xx = 2

zx.basicrules.fuse(g,23+xx,14+xx)
zx.basicrules.fuse(g,23+xx,8)
zx.basicrules.fuse(g,23+xx,32+xx)

zx.basicrules.fuse(g,12,9)
zx.basicrules.fuse(g,21+xx,15+xx)
zx.basicrules.fuse(g,30+xx,24+xx)
zx.basicrules.fuse(g,39+xx,33+xx)

zx.basicrules.fuse(g,13,15)

# This just re-numbers the vertex indexes to remove any gaps:
h = g.copy()
g = h.copy()
gOrig = g.copy()
gEx2 = g.copy()

zx.draw(g, labels=True,scale=30)
print("T-count = ", zx.tcount(g))

T-count =  13


In [12]:
def bestCut(g,tier,vweights_max,vweights,vtiers):
    maxTier = max(vtiers)
    maxW    = -1.0
    bestV   = -1

    for i in range(len(vweights)):
    # for i in vweights.keys():
        w = vweights_max[i]
        if w <= 0 or vtiers[i] != maxTier: continue
        # if (vweights_max[i]>0 and vtiers[i] == maxTier): #if (len(vDoorsteps[i]) > 0):
        if (g.phase(i) in [0.25,0.75,1.25,1.75]): w+=1 # Add 1 to weight if the prospective vertex to cut is itself T-like
        #print(i, "\t", w, "\t", vtiers[i])
        if (w > maxW):
            maxW  = w
            bestV = i
    return bestV

def compWeights(g,showWeights=True):
    tierData     = compFirstTier(g,showWeights)
    tier         = tierData[0]
    vweights_max = tierData[1]
    vweights     = tierData[2]
    vtiers       = tierData[3]
    
    #isNextTier   = True
    #while (isNextTier):
    tierData     = compNextTier(g,showWeights,tier,vweights_max,vweights,vtiers)
    tier         = tierData[0]
    vweights_max = tierData[1]
    vweights     = tierData[2]
    vtiers       = tierData[3]
    #isNextTier   = tierData[5]
    
    vBest        = bestCut(g,tier,vweights_max,vweights,vtiers)
    return [vBest,tier,vweights_max,vweights,vtiers]

In [13]:
print("BEST CUT: vertex",compWeights(g)[0])

--== TIER 1 ==--
vertex 	 weight	 children
10 	 2.0 	 {8, 13}
12 	 1.0 	 {10, 18}
15 	 2.0 	 {16, 13}
18 	 2.0 	 {16, 21}
20 	 3.0 	 {26, 33, 10, 18}
23 	 2.0 	 {24, 21}
26 	 2.0 	 {24, 28}
30 	 2.0 	 {28, 31}
33 	 2.0 	 {34, 31}

--== TIER 2 ==--
12 	 1.0 	 {10, 18}
20 	 3.0 	 {26, 33, 10, 18}

BEST CUT: vertex 20


In [14]:
dispWeights(tier,vweights_max,vweights,vtiers)

vertex	 weight		 tier
10 	 2.0 (+1) 	 1
12 	 1.0 (+0) 	 2
15 	 2.0 (+0) 	 1
18 	 2.0 (+1) 	 1
20 	 3.0 (+1) 	 2
23 	 2.0 (+0) 	 1
26 	 2.0 (+1) 	 1
30 	 2.0 (+0) 	 1
33 	 2.0 (+1) 	 1


##### The weightings are a little different, but we again find that vertex 20 is the best initial cut. So, let's make the cut and decompose our graph into 2 new graphs accordingly...

In [15]:
def cut(v): #TEMP
    x = g.row(v)
    y = g.qubit(v)

    for i in g.neighbors(v):
        newVert = g.add_vertex(2,y,x)
        g.add_edge((i,newVert), 1)

    g.remove_vertex(v)

    zx.draw(g, labels=True)
    #zx.simplify.full_reduce(g)
    #zx.draw(g, labels=True)

#----------

def cut(v): #TEMP
    #for i in g.neighbors(v): g.add_to_phase(25,1,{}) # Add pi to each neighbour
    g.remove_vertex(v)
    
#----------

def pcut(v,p): #TEMP
    # for i in g.neighbors(v): g.add_to_phase(i,0,[p]) # Add param to each neighbour
    for i in g.neighbors(v): g.add_to_phase(i,0) # Add param to each neighbour
    g.remove_vertex(v)
    
#----------

def apply_cut(g: GraphS,v: int) -> list[GraphS, GraphS]: # Return left and right branches of a vertex cut
    gLeft = g.copy()
    gRight = g.copy()
    gLeft.remove_vertex(v)
    gRight.remove_vertex(v)
    
    for i in g.neighbors(v):
        if g.type(i) == 2: # if red
            gRight.set_phase(i,gRight.phase(i)+1)
            
    return [gLeft,gRight]

#----------

def singleFusion(g): # TEMP - this is a very inefficient method
    fusionFound = False
    for i in g.vertices():
        for j in g.neighbors(i):
            if g.type(i) > 0 and g.type(i) == g.type(j) and g.edge_type((i,j)) == 1:
                fusionFound = True
                break
        else: continue
        break
    if fusionFound:
        zx.basicrules.fuse(g,i,j)
    return [g,fusionFound]

def fullFusion(g):
    fullyFused = False
    while fullyFused == False:
        data = singleFusion(g)
        g = data[0]
        fullyFused = not data[1]
    return g

def idRemoval(g: GraphS):
    deadSpiders = set() # Spiders marked for removal

    for i in g.vertices():
        if g.phase(i) == 0 and len(g.neighbors(i)) == 2: # Identity removal
            deadSpiders.add(i)

    for i in deadSpiders:
        neighs = list()
        assert len(g.neighbors(i)) == 2, i
        for neigh in g.neighbors(i): # necessarily only has 2 neighbours here
            neighs.append(neigh)
        g.add_edge((neighs[0],neighs[1]),1)
        g.remove_vertex(i)
    
    return g

In [16]:
zx.draw(g, labels=True, scale=20)
print("=> DECOMPOSES TO...")

cutChoice = 20 # the choice of vertex to cut

gList = apply_cut(g,cutChoice)
zx.draw(gList[0],labels=True,scale=20)
print("+")
zx.draw(gList[1],labels=True,scale=20)

=> DECOMPOSES TO...


+


##### Having applied the cut (and thus decomposed our graph into 2 graphs), let's now partially simplify each while maintining structure. Specifically, we'll look closely at the second one to see how it simplifies...

In [17]:
#TEMP...
g = gList[1].copy()
#g.apply_state("0"*8)  #TEMP
#g.apply_effect("0"*8) #TEMP
zx.draw(g,labels=True,scale=20)

def pi_commute_Z(g: GraphS,v): #zx.basicrules.pi_commute_Z(g,v)
    g.set_phase(v, -g.phase(v))
    ns = g.neighbors(v)
    for w in ns:
        e = g.edge(v, w)
        et = g.edge_type(e)
        if ((g.type(w) == 1 and et == 2) or
            (g.type(w) == 2 and et == 1)):
            # g.add_to_phase(w, 1,[])
            g.add_to_phase(w, 1)
        else:
            g.remove_edge(e)
            c = g.add_vertex(2,
                    qubit=0.5*(g.qubit(v) + g.qubit(w)),
                    row=0.5*(g.row(v) + g.row(w)))
            g.add_edge(g.edge(v, c))
            g.add_edge(g.edge(c, w), edgetype=et)
    return g

def picom(g): # Push any 2-legged pi-phase Z-spiders through into a neighbouring (left-preferred) CNOT
    applied = False
    for v in g.vertices():
        if (g.type(v)==2 and len(g.neighbors(v))==2 and g.phase(v)==1): # if red, 2-legged, and pi-phase
            #for i in g.neighbors(v): if (g.type(i)!=1): #Verify its neighbours are green
            g = pi_commute_Z(g,min(g.neighbors(v)))
            g = idRemoval(g)
            g = fullFusion(g)
            applied = True
            break
    return [g,applied]

def partialSimp(g: GraphS) -> GraphS: # TODO - this function should be made a bit more robust
    g = fullFusion(g)
    g = idRemoval(g)
    g = fullFusion(g)
    
    picomApplies = True
    while (picomApplies == True):
        data = picom(g)
        g = data[0]
        picomApplies = data[1]
        
    g = idRemoval(g)
    return g.copy()

print("...partially simplifies to...")
g = partialSimp(g)
zx.draw(g,labels=True,scale=20)

...partially simplifies to...


##### Now we have two graphs. We can update the weightings on each in a cleverer way, by merging the weights of fused vertices and only recalculating where a change has occurred etc., but since we're not too concerned with super-efficient implementation here, we can simply re-use the functions we have and recompute the weightings (still very quickly/efficiently) on each of these new graphs. And then we just repeat this process and boom! - we have ourselves a decomposition strategy!

##### Let's go back to our initial (example 2) circuit, make it scalar (i.e. plug all its inputs and outputs with something), and see how it fares...

In [18]:
def cutGraphsOld(gList,debug): # Old version (does not ignore redundant, i.e. 0-scalar, terms)
    hList = []
    for i in gList:
        g = i.copy()
        g = partialSimp(g)
        if (debug): zx.draw(g,labels=True,scale=20)
        cutChoice = compWeights(g,debug)[0]
        if (debug): print("     CUT",cutChoice)
        if (cutChoice > -1):
            split = apply_cut(g,cutChoice)
            hList.append(split[0])
            hList.append(split[1])
        else:
            hList.append(g)
    gList = hList.copy()
    return gList
        
def cutGraphs(gList,cliffCount,scalar,debug):
    hList = []
    changed = False
    cutChoices = []
    for i in gList:
        g = i.copy()
        g = partialSimp(g)
        if (debug): zx.draw(g,labels=True,scale=20)
        cutChoice = compWeights(g,debug)[0]
        cutChoices.append(cutChoice)
        if (debug): print("     CUT",cutChoice)
        if (cutChoice > -1):
            split = apply_cut(g,cutChoice)
            
            gTemp0 = split[0].copy(); gTemp0 = partialSimp(gTemp0); zx.simplify.full_reduce(gTemp0)
            if (gTemp0.scalar.is_zero == False and zx.tcount(gTemp0) > 0): hList.append(split[0]) # Only continue cutting non-zero, non-Clifford terms
            else: cliffCount+=1; scalar+=gTemp0.scalar.to_number() # ADD TERM TO GLOBAL SCALAR (and add one to the total number of Clifford terms)
            
            gTemp1 = split[1].copy(); gTemp1 = partialSimp(gTemp1); zx.simplify.full_reduce(gTemp1)
            if (gTemp1.scalar.is_zero == False and zx.tcount(gTemp1) > 0): hList.append(split[1]) # Only continue cutting non-zero, non-Clifford terms
            else: cliffCount+=1; scalar+=gTemp1.scalar.to_number() # ADD TERM TO GLOBAL SCALAR (and add one to the total number of Clifford terms)
            changed = True
        else:
            hList.append(g) # If there are no more cuts found then don't cut (this ideally should never happen)
    gList = hList.copy()
    return [gList,cliffCount,scalar, changed, cutChoices]

In [19]:
g = gOrig.copy()
print("Plug with some states and effects...")
g.apply_state("+"*8)
g.apply_effect("+"*8)
zx.draw(g,scale=20,labels=True)
partialSimp(g)
print("And partially simplify...")
zx.draw(g,scale=20,labels=True)
print("T-count = ", zx.tcount(g))
h = g.copy()

Plug with some states and effects...


And partially simplify...


T-count =  13


In [20]:
debug = False
softDebug = True

cliffCount = 0 # number of clifford terms (i.e. number of leaves of the cutting tree)
scalar     = 0 # total scalar (= sum of the clifford terms)
treeDepth  = 0 # The depth of the cutting tree (i.e. max terms = 2^depth)

g = h.copy()
gTemp = g.copy(); zx.simplify.full_reduce(gTemp)

zx.draw(g, labels=True, scale=20)
print("Initial T-count = ", zx.tcount(g))
print("Reduced T-count = ", zx.tcount(gTemp))

#gList = apply_cut(11)
#zx.draw(gList[0],labels=True,scale=20)
#zx.draw(gList[1],labels=True,scale=20)

gList = [g]
if (debug): print("\nleaves \t buffer\t |  T-counts")
elif (softDebug): print("\nleaves \t buffer")

while(len(gList)>0):
    if (debug): print("##########")
    treeDepth += 1
    data       = cutGraphs(gList,cliffCount,scalar,debug)
    gList      = data[0]
    cliffCount = data[1]
    scalar     = data[2]
    
    if (debug and len(gList) < 10):
        strLine = str(cliffCount) + " \t " + str(len(gList)) + " \t |  "
        for i in range(len(gList)):
            gTemp = gList[i].copy()
            zx.simplify.full_reduce(gTemp)
            strLine += str(zx.tcount(gTemp)) + "  "
            #zx.draw(gTemp,scale=20,labels=True) #TEMP
        print(strLine)
    elif (softDebug): print(cliffCount,"\t",len(gList))

print("\nNUMBER OF CLIFFORD TERMS:",cliffCount)
print("( tree depth:",treeDepth,")")

Initial T-count =  13
Reduced T-count =  11

leaves 	 buffer
0 	 2
0 	 4
8 	 0

NUMBER OF CLIFFORD TERMS: 8
( tree depth: 3 )


##### In this simple case, our initial graph was decomposed into 2 graphs, and each of those was then decomposed into another 2, and each of THOSE into another 2. So, we decomposed our 13 T-count graph into 8 Clifford graph terms, hence alpha ~= 0.23? Well, this is a bit misleading, since our original graph could have been reduced to T-count 11 with Clifford simplification...

In [21]:
g = gOrig.copy()
g.apply_state("+"*8)
g.apply_effect("+"*8)
zx.draw(g,scale=20)
zx.simplify.full_reduce(g)
print("...reduces via Clifford simp to...")
zx.draw(g,scale=20)
h = g.copy()

...reduces via Clifford simp to...


##### So really, applied to this circuit we observe an effective alpha of log_2(8)/11 = 0.27. Still a great result! And much better than the estimate of alpha=0.47 that the BSS decomposition achieves. In fact, let's compare to how well the BSS would have done on this particular circuit...

In [22]:
gDecomp = zx.simulate.find_stabilizer_decomp(g)
print("No. of terms via BSS:",len(gDecomp))

No. of terms via BSS: 7


##### The BSS method (with inter-step Clifford simplification) would have reduced our circuit to 11 stabiliser terms (alpha=0.31). Our method achieved 8 (alpha=0.23)! And this was just a trivial simple example - later we'll see how well it works on much larger T-count circuits. But first, let's check how often our procedure finds the MOST optimal set of vertex cuts (rather than simply AN optimal set)...

## Benchmarking 1 

##### Let's generate a very simple and small example circuit...

In [23]:
strCirc = """
qreg q[3];
ccx q[0], q[1], q[2];
"""

c = zx.qasm(strCirc)
g = c.to_graph()
#zx.draw(g, labels=True)

partialSimp(g)
gOrig = g.copy()

zx.draw(gOrig, labels=True,scale=30)
print("T-count = ", zx.tcount(gOrig))

T-count =  7


##### This is small enough such that we can try out every possible combination of vertex cuts (among its 7 Z-spiders). In each case, we try to simplify as much as possible after the cuts and if any T-spiders still remain then we assume that we can remove them with alpha=0.47 (i.e. falling back on the BSS decomposition)...

In [24]:
debug = False
softDebug = True
testLimit = 500 # set to -1 for no limit

spiders = []
aBest = 999
termsBest = 999999999999

tOrig = zx.tcount(gOrig)
inps = gOrig.inputs()
outs = gOrig.outputs()

for v in gOrig.vertices():
    if (gOrig.type(v) == 1 and not(v in inps) and not(v in outs)): # Only consider Z spiders
        spiders.append(v)
        
for i in range(2**len(spiders)):
    #i = 17954 #TEMP
    g = gOrig.copy()
    
    b = str(bin(i)[2:]).rjust(len(spiders), '0')
    n_cuts = b.count("1") # number of cuts (i.e. 2^n = no. of terms)
    
    for j in range(len(spiders)):
        if(b[j]=="1"): g.remove_vertex(spiders[j])
    
    for j in inps:
        if(len(g.neighbors(j))<1): g.remove_vertex(j)
    for j in outs:
        if(len(g.neighbors(j))<1): g.remove_vertex(j)
    
    zx.simplify.full_reduce(g)
    
    t = zx.tcount(g)
    tDiff = tOrig - t               # tDiff := no. of T-gates that were reduced
    n_terms = 2**n_cuts
    if (t>0): n_terms *= 7**(t/6)  # If there are remaining T-gates, assume these are decomposed via BSS
    a = math.log(n_terms,2)/tOrig   # a := the decomposition efficiency, i.e. alpha
    
    if (debug): zx.draw(g, scale=20, labels=True)
    if (softDebug): print(i, "("+b+")\t|  cuts:", n_cuts, "\t|  ", "Terms:", math.ceil(n_terms), "\t|  ", "ALPHA: ", a)
    if (a < aBest):
        aBest = a
        termsBest = n_terms
    if (testLimit > -1 and i > testLimit): break #TEMP
        
print("\n\nBEST ALPHA:",aBest) # The best result for alpha
print("NUM. TERMS:",termsBest) # The best result for no. of terms (i.e. 2^(aBest)t)

0 (0000000)	|  cuts: 0 	|   Terms: 10 	|   ALPHA:  0.4678924870096007
1 (0000001)	|  cuts: 1 	|   Terms: 2 	|   ALPHA:  0.14285714285714285
2 (0000010)	|  cuts: 1 	|   Terms: 2 	|   ALPHA:  0.14285714285714285
3 (0000011)	|  cuts: 2 	|   Terms: 4 	|   ALPHA:  0.2857142857142857
4 (0000100)	|  cuts: 1 	|   Terms: 2 	|   ALPHA:  0.14285714285714285
5 (0000101)	|  cuts: 2 	|   Terms: 4 	|   ALPHA:  0.2857142857142857
6 (0000110)	|  cuts: 2 	|   Terms: 6 	|   ALPHA:  0.3525560695728001
7 (0000111)	|  cuts: 3 	|   Terms: 12 	|   ALPHA:  0.49541321242994296
8 (0001000)	|  cuts: 1 	|   Terms: 2 	|   ALPHA:  0.14285714285714285
9 (0001001)	|  cuts: 2 	|   Terms: 4 	|   ALPHA:  0.2857142857142857
10 (0001010)	|  cuts: 2 	|   Terms: 4 	|   ALPHA:  0.2857142857142857
11 (0001011)	|  cuts: 3 	|   Terms: 8 	|   ALPHA:  0.42857142857142855
12 (0001100)	|  cuts: 2 	|   Terms: 11 	|   ALPHA:  0.4862396372898289
13 (0001101)	|  cuts: 3 	|   Terms: 8 	|   ALPHA:  0.42857142857142855
14 (0001110)	|  cuts

In [25]:
spiders # The vertices liable for cutting (i.e. 0000011 above means we cut vertices #12 and #14)

[4, 5, 7, 8, 10, 12, 14]

##### Having tried all 128 possible combinations of vertex cuts, we see that the best possible solution(s) removes all the T-gates at the cost of just 2 stabiliser terms (hence alpha = 0.14). For instance, option '1 (0000001)' achieves this result (which cuts only vertex 14)

##### [OPTIONAL] If we install the parameter-supported version of PyZX from https://github.com/mjsutcliffe99/ParamZX, then we can improve the above verifications by including "cut order correction" as outlined in the paper, to correct for any suboptimality in the cut ordering...

In [26]:
def orderCorrect(gg: BaseGraph):
    pRedund = [] # track redundant params
    # for pvars in gg.scalar.phasenodevars:
    for pvars in gg.scalar.phasenodes:
        for var in pvars:
            if not(var in pRedund):
                pRedund.append(var)
                break
    return pRedund

In [27]:
# Demonstrate order correction...

g = gOrig.copy()
zx.draw(g, labels=True,scale=30)

chars = 'abcdefghijklmnopqrstuvwxyz'
pnum = 0

pcut(7,chars[pnum]);  pnum+=1
pcut(14,chars[pnum]); pnum+=1
pcut(4,chars[pnum]);  pnum+=1

for j in inps:
    if(len(g.neighbors(j))<1): g.remove_vertex(j)
for j in outs:
    if(len(g.neighbors(j))<1): g.remove_vertex(j)

zx.draw(g, labels=True,scale=30)

zx.simplify.full_reduce(g)
orderCorrect(g)

[]

##### Above we've considered the option of cutting all 7 Z-spiders in our example graph (which naively results in 2^7 = 128 stabiliser terms). However, with some parameter analysis, we can infer that we have one redundant parameter ('b') and so (ignoring the 0-terms) this would actually result in 2^6 = 64 stabiliser terms. Let's look at another example...

In [28]:
# Demonstrate order correction...

g = gEx2.copy()
zx.draw(g, labels=True, scale=30)

spiders = []
t = zx.tcount(g)
inps = g.inputs()
outs = g.outputs()

chars = 'abcdefghijklmnopqrstuvwxyz'
pnum = 0

pcut(10,chars[pnum]); pnum+=1
pcut(18,chars[pnum]); pnum+=1
pcut(26,chars[pnum]); pnum+=1
pcut(33,chars[pnum]); pnum+=1
#pcut(12,chars[pnum]); pnum+=1
pcut(20,chars[pnum]); pnum+=1

for j in inps:
    if(len(g.neighbors(j))<1): g.remove_vertex(j)
for j in outs:
    if(len(g.neighbors(j))<1): g.remove_vertex(j)

zx.draw(g, labels=True, scale=30)

zx.simplify.full_reduce(g)
orderCorrect(g)

[]

##### In this graph, if we were to cut vertices 10, 18, 26, 33, and 20, we'd naively conclude this results in 2^5 = 32 terms. However, with some cut order correction we see that we actually have 2 redundant parameters here ('b' and 'c') and hence this would actually reduce to 2^3 = 7 terms (as if we had cut vertex 20 first and THEN fused the others and cut them as one).

##### So, let's redo our verification experiment from above, but with this cut order correction...

In [29]:
# PARAM-SUPPORTED VERSION...

debug = False
softDebug = True
doOrderCorrect = True # Use parameter analysis to correct for any suboptimal cut ordering
testLimit = 500 # set to -1 for no limit

spiders = []
aBest = 999
termsBest = 999999999999
chars = 'abcdefghijklmnopqrstuvwxyz'

tOrig = zx.tcount(gOrig)
inps = gOrig.inputs()
outs = gOrig.outputs()

for v in gOrig.vertices():
    if (gOrig.type(v) == 1 and not(v in inps) and not(v in outs)): # Only consider Z spiders
        spiders.append(v)
        
for i in range(2**len(spiders)):
    #i = 17954 #TEMP
    g = gOrig.copy()
    pnum = 0
    
    b = str(bin(i)[2:]).rjust(len(spiders), '0')
    n_cuts = b.count("1") # number of cuts (i.e. 2^n = no. of terms)
    
    for j in range(len(spiders)):
        if(b[j]=="1"):
            pcut(spiders[j],chars[pnum]) #g.remove_vertex(spiders[j])
            pnum += 1
    
    for j in inps:
        if(len(g.neighbors(j))<1): g.remove_vertex(j)
    for j in outs:
        if(len(g.neighbors(j))<1): g.remove_vertex(j)
    
    zx.simplify.full_reduce(g)
    for v in g.vertices(): g.set_phase(v,g.phase(v)) # Reduce all params -> 0
    
    if (doOrderCorrect):
        n_cuts -= len(orderCorrect(g)) # Correct for suboptimal cut order
        #print("cut correct",orderCorrect(g)) #TEMP
    zx.simplify.full_reduce(g)
    
    t = zx.tcount(g)
    tDiff = tOrig - t               # tDiff := no. of T-gates that were reduced
    n_terms = 2**n_cuts
    if (t>0): n_terms *= 7**(t/6)  # If there are remaining T-gates, assume these are decomposed via BSS
    a = math.log(n_terms,2)/tOrig   # a := the decomposition efficiency, i.e. alpha
    
    if (debug): zx.draw(g, scale=20, labels=True)
    if (softDebug): print(i, "("+b+")\t|  cuts:", n_cuts, "\t|  ", "Terms:", math.ceil(n_terms), "\t|  ", "ALPHA: ", a)
    if (a < aBest):
        aBest = a
        termsBest = n_terms
    if (testLimit > -1 and i > testLimit): break #TEMP
        
print("\n\nBEST ALPHA:",aBest) # The best result for alpha
print("NUM. TERMS:",termsBest) # The best result for no. of terms (i.e. 2^(aBest)t)

0 (0000000)	|  cuts: 0 	|   Terms: 10 	|   ALPHA:  0.4678924870096007
1 (0000001)	|  cuts: 1 	|   Terms: 2 	|   ALPHA:  0.14285714285714285
2 (0000010)	|  cuts: 1 	|   Terms: 2 	|   ALPHA:  0.14285714285714285
3 (0000011)	|  cuts: 2 	|   Terms: 4 	|   ALPHA:  0.2857142857142857
4 (0000100)	|  cuts: 1 	|   Terms: 2 	|   ALPHA:  0.14285714285714285
5 (0000101)	|  cuts: 2 	|   Terms: 4 	|   ALPHA:  0.2857142857142857
6 (0000110)	|  cuts: 2 	|   Terms: 6 	|   ALPHA:  0.3525560695728001
7 (0000111)	|  cuts: 3 	|   Terms: 12 	|   ALPHA:  0.49541321242994296
8 (0001000)	|  cuts: 1 	|   Terms: 2 	|   ALPHA:  0.14285714285714285
9 (0001001)	|  cuts: 2 	|   Terms: 4 	|   ALPHA:  0.2857142857142857
10 (0001010)	|  cuts: 2 	|   Terms: 4 	|   ALPHA:  0.2857142857142857
11 (0001011)	|  cuts: 3 	|   Terms: 8 	|   ALPHA:  0.42857142857142855
12 (0001100)	|  cuts: 2 	|   Terms: 11 	|   ALPHA:  0.4862396372898289
13 (0001101)	|  cuts: 3 	|   Terms: 8 	|   ALPHA:  0.42857142857142855
14 (0001110)	|  cuts

##### The final "most optimal" result is the same in this case, but looking at the results for specific combinations we see that this actually results in much fewer terms in most cases (as we're essentially ignoring redundant branches when cutting).

##### We can replace the example circuit above with any other, or generate a bunch of random examples to benchmark our method and see how often our approach finds the most optimal result

## Benchmarking 2

##### Now let's consider some larger random circuits (which are too big to verify with brute force as above). Instead, we want to compare our results to the method of Kissinger and van de Wetering (which relies primarily on the BSS decomposition). So, first let's generate a large random circuit that has some local structural elements...

##### [Note: set useExample = False here if you want to generate new random examples, rather than using the one generated in advance. But note that the text ahead will refer to the results from this particular example circuit]

In [30]:
useExample = False # FEEL FREE TO TOGGLE THIS

In [31]:
# GENERATE RANDOM (NON-TRIVIALLY) STRUCTURED CIRCUITS...

includeToffHads = True # Include the Hadamards in the Toffoli?

def rq(qasmLines: list, NQ: int, avoidA=-1, avoidB=-1): # select a random qubit (avoiding, if desired, specific ones)
    if (avoidA < 0 and avoidB < 0): return random.randrange(1,NQ+1)-1
    rNum = avoidA
    while (rNum in [avoidA, avoidB]): rNum = random.randrange(1,NQ+1)-1
    return rNum

def addGate(qasmLines: list, NQ: int, gate, b=-1, a=-1, c=-1): # Add a gate on qubits a,b,c (or randomise either/both, with a!=b!=c, if unspecified)
    if (b<0): b=rq(qasmLines, NQ)
    if (a<0): a=rq(qasmLines, NQ, b)
    if (c<0): c=rq(qasmLines, NQ, a,b)
    strGate = "\n"
    match gate:
        case "t":    strGate += "t q[" + str(b) + "];"
        case "cnot": strGate += "cx q[" + str(a) + "], q[" + str(b) + "];"
        case "toff":
            if (includeToffHads): strGate += "h q[" + str(c) + "];\n"
            strGate += "ccz q[" + str(a) + "], q[" + str(b) + "], q[" + str(c) + "];"
            if (includeToffHads): strGate += "\nh q[" + str(c) + "];"
        case "rz":
            ph = random.randrange(1,8) # random Clifford+T phase (no point in including 0)
            if (b==1): ph = random.randrange(1,4)*2 # if Clifford
            ph = ph/4
            strGate += "rz(" + str(ph) + "*pi) q[" + str(a) + "];"
        case "rx":
            ph = random.randrange(1,8) # random Clifford+T phase (no point in including 0)
            if (b==1): ph = random.randrange(1,4)*2 # if Clifford
            ph = ph/4
            strGate += "rx(" + str(ph) + "*pi) q[" + str(a) + "];"
    qasmLines.append(strGate)
    return [a,b,c]

def tSandwich(qasmLines: list, NQ: int, t=2): # Generate a T-CNOT-T-CNOT-T-... sandwich of length t (minimum t=2)
    q = addGate(qasmLines, NQ, "t")[1]
    for i in range(t-1):
        addGate(qasmLines, NQ, "cnot",q)
        addGate(qasmLines, NQ, "t",q)
        
def toff(qasmLines:list, NQ:int, n=1,cnots=False): # Generate a chain of randomly placed Toffoli gates ('cnots' = whether to separate each with a cnot)
    for i in range(n):
        q = addGate(qasmLines,NQ, "toff")[0]
        if(cnots): addGate(qasmLines,NQ, "cnot", q)
            
def rz(qasmLines,NQ, cliff=True): addGate(qasmLines,NQ, "rz",int(cliff))
def rx(qasmLines,NQ, cliff=True): addGate(qasmLines,NQ, "rx",int(cliff))

In [32]:
# Example (feel free to change this up)...

NQ = 8 # No. of qubits

strCirc = "\nqreg q[" + str(NQ) + "];"
qasmLines = list()

tSandwich(qasmLines, NQ, 4)
toff(qasmLines, NQ, 2,True)
rz(qasmLines, NQ, True)
toff(qasmLines, NQ, 1,True)
rz(qasmLines, NQ, False)
toff(qasmLines, NQ, 1,True)
rz(qasmLines, NQ, False)
rz(qasmLines, NQ, False)

#----------

for line in qasmLines: strCirc += line
if (useExample): strCirc = "\nqreg q[6];\nt q[4];\ncx q[2], q[4];\nt q[4];\ncx q[0], q[4];\nt q[4];\ncx q[1], q[4];\nt q[4];\nh q[3];\nccz q[2], q[4], q[3];\nh q[3];\ncx q[0], q[2];\nh q[0];\nccz q[3], q[2], q[0];\nh q[0];\ncx q[2], q[3];\nrz(0.5*pi) q[2];\nh q[1];\nccz q[4], q[0], q[1];\nh q[1];\ncx q[1], q[4];\nh q[3];\nccz q[2], q[4], q[3];\nh q[3];\ncx q[4], q[2];\nh q[1];\nccz q[0], q[4], q[1];\nh q[1];\ncx q[3], q[0];\nh q[5];\nccz q[1], q[0], q[5];\nh q[5];\ncx q[4], q[1];\nt q[0];\ncx q[1], q[0];\nt q[0];\ncx q[4], q[0];\nt q[0];\nrz(0.5*pi) q[0];"
c = zx.qasm(strCirc)
g = c.to_graph()

g.apply_state("+"*NQ)  #TEMP
g.apply_effect("+"*NQ) #TEMP

#g.normalize()
zx.draw(g, labels=True, scale=20)
print("T-count = ", zx.tcount(g))

T-count =  33


##### Here we've generated a random circuit that has a T-count of 49 (unless you generated a new random circuit). With some partial simplification (but keeping the graph structure), this reduces a little to 47 in this case...

In [33]:
partialSimp(g)
pseudog = g.copy()
h = g.copy(); g = h.copy() # This just re-numbers the vertex indexes to remove any gaps
zx.draw(g, labels=True, scale=30)
print("T-count = ", zx.tcount(g))

T-count =  33


##### Even though we'll be starting from this above circuit (without doing a full Clifford simplification first - as this compromises the structure), we need to know what the TRUE initial T-count is (after applying full Clifford simp.)...

In [34]:
zx.simplify.full_reduce(g)
zx.draw(g, labels=True, scale=30)
print("T-count = ", zx.tcount(g))
g = h.copy()

T-count =  0


##### 22. So we'll take note of this as our initial T-count (the best we can get before needing to decompose something)

##### Now, let's go back to our graph-like version and apply the procedure to it...

In [35]:
def killnullverts(gg): # This removes any free (legless) nodes of null-type (i.e. the little black dots)
    nullverts = list()
    for v in gg.vertices():
        if (g.type(v)==0 and len(g.neighbors(v))==0): nullverts.append(v)
    for v in nullverts:
        gg.remove_vertex(v)

In [36]:
vData = compWeights(g)

vBest        = vData[0]
tier         = vData[1]
vweights_max = vData[2]
vweights     = vData[3]
vtiers       = vData[4]

--== TIER 1 ==--
vertex 	 weight	 children
0 	 5.0 	 {4, 8, 22, 26, 28, 30}
2 	 8.0 	 {6, 8, 10, 14, 16, 18, 20, 22}
3 	 6.0 	 {32, 0, 34, 26, 28, 30}
5 	 6.0 	 {36, 38, 40, 42, 44, 46}
6 	 4.0 	 {16, 18, 12, 14}
24 	 2.0 	 {50, 3}
36 	 3.0 	 {40, 2, 51, 42}
38 	 1.0 	 {2, 51}
48 	 6.0 	 {51, 52, 54, 56, 58, 60}
51 	 3.0 	 {56, 34, 52, 54}

--== TIER 2 ==--
36 	 1.0 	 {2, 51}
38 	 1.0 	 {2, 51}



In [37]:
dispWeights(tier,vweights_max,vweights,vtiers)
print("\nBEST CUT:",vBest)

vertex	 weight		 tier
0 	 5.0 (+1) 	 1
2 	 8.0 (+1) 	 1
3 	 6.0 (+1) 	 1
5 	 6.0 (+1) 	 1
6 	 4.0 (+1) 	 1
24 	 2.0 (+0) 	 1
36 	 3.0 (+1) 	 2
38 	 1.0 (+1) 	 2
48 	 6.0 (+1) 	 1
51 	 3.0 (+1) 	 1

BEST CUT: 36


In [38]:
def getTierCutOrder(tier,vweights_max,vweights,vtiers): # Returns the cut order by weight, for a specific tier
    VS = []
    WS = []
    for i in range(len(vweights)):
        if (vtiers[i] == tier and vweights_max[i]>0): #if (len(vDoorsteps[i]) > 0):
            w = vweights_max[i]
            b = 0
            if (g.phase(i) in [0.25,0.75,1.25,1.75]): b+=1 # Add 1 to weight if the prospective vertex to cut is itself T-like
            VS.append(i)
            WS.append(w+b)
    return [VS,WS]
            
def getCutOrder(vweights_max,vweights,vtiers): # Returns ALL weighted vertices, ordered by cutting priority (best cut to worst)
    cutOrders = []
    for t in range(max(vtiers),0,-1):
        data = getTierCutOrder(t,vweights_max,vweights,vtiers)
        VS = data[0]
        WS = data[1]
        cutOrder = [x for _,x in sorted(zip(WS,VS))]
        cutOrder.reverse()
        cutOrders += cutOrder
    return cutOrders

In [39]:
cutOrder = getCutOrder(vweights_max,vweights,vtiers)
print(cutOrder)

[36, 38, 2, 48, 5, 3, 0, 6, 51, 24]


##### In the paper, we present a method of efficiently/quickly estimating the number of terms produced by this method, utilising parameterisation. Here, we offer a (slightly less automatic) alternative estimation approach which does not require the modified PyZX package (but rather can run on the standard version of PyZX)...

##### The above (getCutOrder) function gives us a list of the weighted vertices in order of cutting priority. For a quick ESTIMATION of the number of stabiliser terms we're likely to get, we'll assume this list does not change after the cuts, or vary between branches (as in reality it would). So, let's decide how many we want to cut (starting from the highest priority down)...

In [40]:
cutcount = 4 # FEEL FREE TO CHANGE THIS

In [41]:
# CUT ANY NUMBER OF VERTICES, BY ORDER OF CUTTING PREFERENCE (AS DETERMINED BY THE PROCEDURE)...

g = h.copy()
zx.draw(g, labels=True, scale=20)
print("T-count = ", zx.tcount(g), "\n\n")

cutOrder = getCutOrder(vweights_max,vweights,vtiers)

for i in range(cutcount):
    g.remove_vertex(cutOrder[i]); killnullverts(g);

#partialSimp()
zx.simplify.full_reduce(g)

print("cutcount:",cutcount,"     |  ",2**(cutcount))
zx.draw(g, labels=True, scale=20)
print("T-count = ", zx.tcount(g))

T-count =  33 


cutcount: 4      |   16


T-count =  0


##### We can use a bit of trial and error to find that cutcount = 4 seems to be the best choice - i.e. the fewest number of cuts that reduces our circuit to a trivially small T-count (at T-count 6 we can use BSS).

In [42]:
#TODO...
# BSS DECOMP. WHAT REMAINS...

gT = zx.tcount(g)
gDecomp = zx.simulate.find_stabilizer_decomp(g)

print("CUT CONTRIBUTION...")
print("Exact:   ",2**cutcount)

print("\nBSS CONTRIBUTION...")
print("Exact:   ",len(gDecomp))
print("Estimate:",7**(gT/6))

print("\nTOTAL...")
print("Exact:   ",(2**cutcount)*len(gDecomp))
print("Estimate:",(2**cutcount)*(7**(gT/6)))

CUT CONTRIBUTION...
Exact:    16

BSS CONTRIBUTION...
Exact:    1
Estimate: 1.0

TOTAL...
Exact:    16
Estimate: 16.0


##### So, we made 4 cuts (hence 2^4 = 16 terms), followed by one instance of the BSS (which split each into another 4). So, we estimate that we end up with 64 stabiliser terms. Recall that this was to reduce a circuit of T-count 22, hence this achieves an alpha = 0.27.

##### This was our estimate (which we could compute very rapidly, regardless of circuit depth). Let's actually try it out for real and see what we get...

In [43]:
#Tthreshold = 7 # if a term has <= this many T-gates remaining then we fall back on BSS

debug = False
softDebug = True

cliffCount = 0 # number of clifford terms (i.e. number of leaves of the cutting tree)
scalar     = 0 # total scalar (= sum of the clifford terms)
treeDepth  = 0 # The depth of the cutting tree (i.e. max terms = 2^depth)

g = h.copy()
gTemp = g.copy(); zx.simplify.full_reduce(gTemp)

zx.draw(g, labels=True, scale=20)
print("Initial T-count = ", zx.tcount(g))
print("Reduced T-count = ", zx.tcount(gTemp))

#gList = apply_cut(11)
#zx.draw(gList[0],labels=True,scale=20)
#zx.draw(gList[1],labels=True,scale=20)

gList = [g]
if (debug): print("\nleaves \t buffer\t |  T-counts")
elif (softDebug): print("\nleaves \t buffer")

while(len(gList)>0):
    treeDepth += 1
    data       = cutGraphs(gList,cliffCount,scalar,False)
    gList      = data[0]
    cliffCount = data[1]
    scalar     = data[2]
    
    if (debug and len(gList) < 10):
        strLine = str(cliffCount) + " \t " + str(len(gList)) + " \t |  "
        for i in range(len(gList)):
            gTemp = gList[i].copy()
            zx.simplify.full_reduce(gTemp)
            strLine += str(zx.tcount(gTemp)) + "  "
            zx.draw(gTemp,scale=20,labels=True) #TEMP
        print(strLine)
    elif (softDebug): print(cliffCount,"\t",len(gList))

print("\nNUMBER OF CLIFFORD TERMS:",cliffCount)
print("( tree depth:",treeDepth,")")

Initial T-count =  33
Reduced T-count =  0

leaves 	 buffer
0 	 2
2 	 2
6 	 0

NUMBER OF CLIFFORD TERMS: 6
( tree depth: 3 )


In [44]:
def find_decomp_ori(g: BaseGraph) -> tuple[int, complex, list[list[int]]]:
    treeDepth = 0
    gList = [g.copy()]
    cliffCount = 0
    scalar = complex(0)

    cutList = []
    while(len(gList)>0):
        treeDepth += 1
        data       = cutGraphs(gList,cliffCount,scalar,False)
        # print(f"{data = }")
        if not data[3]:
            hList = []
            for g in data[0]:
                gt = g.copy()
                zx.simplify.full_reduce(gt)
                gsum = zx.simulate.replace_magic_states(gt, True)
                gsum.reduce_scalar() 
                hList.extend(gsum.graphs)
            # print(f"{len(hList) = }, {cliffCount = }")
            gList = []
            for g in hList:
                # print(f"{zx.simplify.tcount(g) = }")
                gt = g.copy(); zx.simplify.full_reduce(gt)
                # print(f"{zx.simplify.tcount(gt) = }")
                if gt.scalar.is_zero: continue

                if zx.simplify.tcount(gt) > 0 :
                    gList.append(g)
                else: 
                    cliffCount += 1
                    scalar += g.scalar.to_number()

            # print(f"{len(gList) = }, {cliffCount = }")
            cutList.append(-1)
        else:
            gList = data[0]
            cliffCount = data[1]
            scalar     = data[2]
            cutList.append(data[4])
    # print(f"{treeDepth = }")
    return cliffCount, scalar, cutList

##### And there we have it! We ended up with 31 stabiliser terms! In reducing a circuit of T-count 22, this achieves an alpha = 0.225 - very good!

##### And for comparison, let's see what the Kissinger/Wetering (BSS-based) approach would have achieved for this circuit...

In [45]:
# %%time
g = h.copy()

#g.apply_state("+"*6)  ## TEMP
#g.apply_effect("+"*6) ## TEMP

zx.draw(g, scale=20, labels=True)
print("T-count = ", zx.tcount(g))

zx.simplify.full_reduce(g)

zx.draw(g, scale=20, labels=True)
print("T-count = ", zx.tcount(g))
gT = zx.tcount(g)

gDecomp = zx.simulate.find_stabilizer_decomp(g)
print("\nExact:   ",len(gDecomp))
print("Estimate:",7**(gT/6))

g = h.copy()

T-count =  33


T-count =  0

Exact:    1
Estimate: 1.0


In [46]:
# from pyzx.simulate import find_stabilizer_decomp_cat

# g = gOrig.copy()
# zx.full_reduce(g)
# gDecompCat = find_stabilizer_decomp_cat(g)
# print("\nExact:   ",len(gDecompCat))
# print("Estimate:",7**(gT/6))

##### 303 terms! Almost 10 times as many terms, with an alpha = 0.37, as compared to our alpha = 0.225.

In [47]:
def bestCutM(g: GraphS,tier,vweights_max: list[int],vweights: list[int],vtiers: list[int], returnBest: bool = True):
    maxTier = max(vtiers)
    maxW    = -1.0
    bestV   = -1

    # List of posible vertices
    possV = []

    # for i in vweights.keys():
    for i in range(len(vweights)):

        w = vweights_max[i]
        if w <= 0 or vtiers[i] != maxTier: continue
        possV.append(i)

        if (g.phase(i) in [0.25,0.75,1.25,1.75]): w+=1 # Add 1 to weight if the prospective vertex to cut is itself T-like
        #print(i, "\t", w, "\t", vtiers[i])
        vweights_max[i] = w
        if (w > maxW):
            maxW  = w
            bestV = i

    if returnBest: return bestV

    return possV, bestV

def compWeightsM(g: GraphS,showWeights: bool=True) -> tuple[int, int, list[int], list[int], list[int]]:
    tierData     = compFirstTier(g,showWeights)
    tier         = tierData[0]
    vweights_max = tierData[1]
    vweights     = tierData[2]
    vtiers       = tierData[3]
    
    #isNextTier   = True
    #while (isNextTier):
    tierData     = compNextTier(g,showWeights,tier,vweights_max,vweights,vtiers)
    tier         = tierData[0]
    vweights_max = tierData[1]
    vweights     = tierData[2]
    vtiers       = tierData[3]
    #isNextTier   = tierData[5]
    
    # vBest        = bestCutM(g,tier,vweights_max,vweights,vtiers)
    possV, vBest = bestCutM(g,tier,vweights_max,vweights,vtiers, False)
    return [vBest,tier,vweights_max,vweights,vtiers, possV]

In [48]:
def cutGraph(g: GraphS, v: int) -> tuple[list[GraphS], int, complex]:
    gList = []
    cliffCount = 0
    scalar = 0.0
    g0, g1 = apply_cut(g,v)
    gTemp0 = partialSimp(g0.copy()); zx.simplify.full_reduce(gTemp0)
    gTemp1 = partialSimp(g1.copy()); zx.simplify.full_reduce(gTemp1)
    
    if (not gTemp0.scalar.is_zero and zx.tcount(gTemp0) > 0): gList.append(g0) # Only continue cutting non-zero, non-Clifford terms
    else: cliffCount+=1; scalar+=gTemp0.scalar.to_number() # ADD TERM TO GLOBAL SCALAR (and add one to the total number of Clifford terms)
    
    if (not gTemp1.scalar.is_zero and zx.tcount(gTemp1) > 0): gList.append(g1) # Only continue cutting non-zero, non-Clifford terms
    else: cliffCount+=1; scalar+=gTemp1.scalar.to_number() # ADD TERM TO GLOBAL SCALAR (and add one to the total number of Clifford terms)

    return gList, cliffCount, scalar

In [49]:
# import heapq

# def cutGraph_AStar(g0: GraphS, cliffCount: int, scalar: complex, debug: bool):
#     max_depth = 10
#     d = 0
#     max_actions = 5
#     pq = []
#     gList = []
#     g0 = g0.copy()
#     g0 = partialSimp(g0)
#     if (debug): zx.draw(g0,labels=True,scale=20)

#     # If there are no more cuts found then don't cut (this ideally should never happen)
#     if vBest < 0:
#         return [g, cliffCount, scalar]
    
#     # Put weights in a priority queue
#     # (weight, vertex, graph, depth)
#     # pq = [(-w, v, [g.copy()], 0) for v, w in enumerate(vweights_max)]
#     # heapq.heapify(pq)
#     # Only consider best actions
#     # (weight, depth, graphList)
#     pq.append((0, 0, [g]))

#     while pq:
#         w, d, gList = heapq.heappop(pq)

#         for h in gList:

#             g = h.copy()
#             g = partialSimp(g)
#             if (debug): zx.draw(g,labels=True,scale=20)
#             vBest,tier,vweights_max,vweights,vtiers = compWeights_AStar(g,debug)
#             chosenCuts = [(w, v) for v, w in enumerate(vweights_max)]
#             chosenCuts.sort()
#             chosenCuts = [v for w, v in chosenCuts[:max_actions]]

#             for i in chosenCuts:
#                 hList, hCliffCount, hScalar = cutGraph(g, i)
        
        
#         # g0, g1 = apply_cut(g,v)
#         # gTemp0 = partialSimp(g0.copy()); zx.simplify.full_reduce(gTemp0)
#         # gTemp1 = partialSimp(g1.copy()); zx.simplify.full_reduce(gTemp1)
        
#         # if (not gTemp0.scalar.is_zero and zx.tcount(gTemp0) > 0): gList.append(g0) # Only continue cutting non-zero, non-Clifford terms
#         # else: cliffCount+=1; scalar+=gTemp0.scalar.to_number() # ADD TERM TO GLOBAL SCALAR (and add one to the total number of Clifford terms)
        
#         # if (not gTemp1.scalar.is_zero and zx.tcount(gTemp1) > 0): gList.append(g1) # Only continue cutting non-zero, non-Clifford terms
#         # else: cliffCount+=1; scalar+=gTemp1.scalar.to_number() # ADD TERM TO GLOBAL SCALAR (and add one to the total number of Clifford terms)

#     return [gList,cliffCount,scalar]

##### Evidently, our procedural method is very effective! For the paper, we repeated this last section ("Benchmarking 2") for many random circuits of various depths and found that our method was consistently magnitudes better than the BSS approach - achieving typically alpha between 0.1 and 0.2 (as compared to ~0.4 with the BSS method) :D

In [50]:
# def cutGraphs_AStar(gList,cliffCount,scalar,debug):
#     hList = []
#     for i in gList:
#         g = i.copy()
#         g = partialSimp(g)
#         if (debug): zx.draw(g,labels=True,scale=20)
#         cutChoice = compWeights(g,debug)[0]
#         if (debug): print("     CUT",cutChoice)
#         if (cutChoice > -1):
#             split = apply_cut(g,cutChoice)
            
#             gTemp0 = split[0].copy(); gTemp0 = partialSimp(gTemp0); zx.simplify.full_reduce(gTemp0)
#             if (gTemp0.scalar.is_zero == False and zx.tcount(gTemp0) > 0): hList.append(split[0]) # Only continue cutting non-zero, non-Clifford terms
#             else: cliffCount+=1; scalar+=gTemp0.scalar.to_number() # ADD TERM TO GLOBAL SCALAR (and add one to the total number of Clifford terms)
            
#             gTemp1 = split[1].copy(); gTemp1 = partialSimp(gTemp1); zx.simplify.full_reduce(gTemp1)
#             if (gTemp1.scalar.is_zero == False and zx.tcount(gTemp1) > 0): hList.append(split[1]) # Only continue cutting non-zero, non-Clifford terms
#             else: cliffCount+=1; scalar+=gTemp1.scalar.to_number() # ADD TERM TO GLOBAL SCALAR (and add one to the total number of Clifford terms)
#         else:
#             hList.append(g) # If there are no more cuts found then don't cut (this ideally should never happen)
#     gList = hList.copy()
#     return [gList,cliffCount,scalar]

In [51]:
# class GraphNode:
#     def __init__(self, graph: GraphS):
#         self.graph = graph
#         self.sorted_cuts = self.get_sorted_cuts()

#     def get_sorted_cuts(self):
#         vBest,tier,vweights_max,vweights,vtiers = compWeights(g,debug)
#         chosenCuts = [(w, v) for v, w in enumerate(vweights_max)]
#         chosenCuts.sort()
#         return chosenCuts

In [52]:
import numpy as np
from itertools import product

class Node:
    def __init__(self, graphs: list[GraphS], parent=None, vertex_cut_list: list[int]=None, 
                 cliffCount: int=None, scalar: complex=None, depth:int=None):
        self.graphs = graphs  # The current state of the ZX-diagram
        self.parent = parent  # Parent node
        # self.graph_cut_id = graph_cut_id # Graph cut to reach this node
        self.vertex_cut_list = vertex_cut_list  # Vertices cut to reach this node
        self.children = []  # List of child nodes
        self.score = 0.0  # Number of terms
        self.visits = 0  # Number of visits during simulations
        self.actions_done = set()
        self.max_vertices = 2
        self.max_actions = 10
        self.scalar = complex(0) if scalar is None else scalar
        self.cliffCount = 0 if cliffCount is None else cliffCount
        self.depth = 0 if depth is None else depth
        self.untried_actions = None
        self.best_action = None
        self.all_actions = None

    def __len__(self):
        return len(self.graphs)

    def is_fully_expanded(self):
        # Assuming a function that returns all possible cuts
        if self.is_terminal(): return True
        # return len(self.children) >= len(self) * self.max_vertices
        # return len(self.children) >= self.max_vertices
        # return False
        # return len(self.children) == len(self.diagram.vertices())
        untried_actions = self.get_untried_actions()
        if not untried_actions: 
            return len(self.children) >= 1
        return len(self.children) >= self.max_actions


    def best_child(self, c_param=1.4):
        choices_weights = [
            (child.score / child.visits) + c_param * np.sqrt((2 * np.log(self.visits) / child.visits))
            for child in self.children
        ]
        # print(choices_weights)
        return self.children[np.argmax(choices_weights)]
    
    def is_terminal(self):
        return not self.graphs
    
    def get_all_actions(self):
        if self.all_actions is not None: return self.all_actions
        vList = []
        bList = []
        for g in self.graphs:
            g = g.copy()
            
            vBest,tier,vweights_max,vweights,vtiers, possV = compWeightsM(g,False)
            
            orderedV = sorted([(w, v) for v, w in enumerate(vweights_max) if w > 0], reverse=True)
            possV = [v for w, v in orderedV]
            vList.append(tuple(possV[:self.max_vertices]))
            bList.append(vBest)
        
        self.best_action = tuple(bList)
        self.all_actions = set(product(*vList))
        # self.all_actions.add(-1) # representing BSS
        # print(self.all_actions)
        self.untried_actions = self.all_actions.copy()
        return self.all_actions
    
    def get_untried_actions(self):
        if self.untried_actions is not None: return self.untried_actions
        return self.get_all_actions()
    
    def get_best_action(self):
        if self.best_action is not None: return self.best_action
        self.get_all_actions()
        return self.best_action

returnBest = False

MAX_VERTICES = 5
def get_random_actions(gList: list[GraphS], returnBest: bool = True) -> list[int]:
    
    vList = []
    for g in gList:
        g = g.copy()

        # iGraph = random.randrange(len(self))
        # g = self.graphs[iGraph]

        # vBest,tier,vweights_max,vweights,vtiers = compWeights(g,False)
        vBest,tier,vweights_max,vweights,vtiers, possV = compWeightsM(g,False)
        # chosenCuts = [(w, v) for v, w in enumerate(vweights_max)]
        # chosenCuts.sort(reverse=True)
        # chosenCuts = [v for w, v in chosenCuts[:MAX_VERTICES]]
        # chosenCuts = sorted()
        # chosenCuts = list((v, vweights_max[v]) for v in possV)
        v = vBest
        if not returnBest and vBest > -1:
            possW = [vweights_max[v] for v in possV]
            sW = sum(possW)
            possW = [w/sW for w in possW]
            v = random.choices(possV, weights=possW, k=1)[0]
        
        vList.append(v)
    return vList
    
def cutGraphMCTS(g: GraphS, v: int) -> tuple[list[GraphS], int, complex]:
    gList = []
    cliffCount = 0
    scalar = complex(0)
    g0, g1 = apply_cut(g,v)
    g0 = partialSimp(g0.copy())
    g1 = partialSimp(g1.copy())
    gTemp0 = g0.copy(); zx.simplify.full_reduce(gTemp0)
    gTemp1 = g1.copy(); zx.simplify.full_reduce(gTemp1)
    
    if (not gTemp0.scalar.is_zero and zx.tcount(gTemp0) > 0): gList.append(g0) # Only continue cutting non-zero, non-Clifford terms
    else: cliffCount+=1; scalar+=gTemp0.scalar.to_number() # ADD TERM TO GLOBAL SCALAR (and add one to the total number of Clifford terms)
    
    if (not gTemp1.scalar.is_zero and zx.tcount(gTemp1) > 0): gList.append(g1) # Only continue cutting non-zero, non-Clifford terms
    else: cliffCount+=1; scalar+=gTemp1.scalar.to_number() # ADD TERM TO GLOBAL SCALAR (and add one to the total number of Clifford terms)
    
    return gList, cliffCount, scalar

def cutGraphsMCTS(gList: list[GraphS], cliffCount: int=0, scalar: complex=complex(0), vList: list[int]=None) \
    -> tuple[list[GraphS], int, complex, bool]:
    # vList = node.get_random_actions()
    # gList = node.graphs
    hList = []
    # scalar = complex(0)
    # cliffCount = 0
    changed = False
    for i, h in enumerate(gList):
        h = h.copy()
        # g = h.copy()
        # g = partialSimp(g)
        # if (debug): zx.draw(g,labels=True,scale=20)
        cutChoice = vList[i]
        # if (debug): print("     CUT",cutChoice)

        # print(f"{len(g.vertices()) = }, {cutChoice = }")

        # If there are no more cuts found then don't cut (this ideally should never happen)
        if cutChoice <= -1:
            hList.append(h)
            continue

        changed = True
        try:
            gCutList, addCliffCount, addScalar = cutGraphMCTS(h, cutChoice)
        except AssertionError as e:
            print(f"error cutting {cutChoice = } for graph {i}")
            zx.draw(h,scale=20,labels=True)
            raise AssertionError(e)
        hList.extend(gCutList)
        cliffCount += addCliffCount
        scalar += addScalar
    return hList, cliffCount, scalar, changed


def selection(node: Node) -> Node:
    while not node.is_terminal():
        if not node.is_fully_expanded():
            return expansion(node)
        else:
            node = node.best_child()
    return node



def expansion(node: Node) -> Node:
    # Assuming a function that returns untried actions for the node
    untried_actions = tuple(node.get_untried_actions())
    vList = random.choice(untried_actions) if untried_actions else -1
    
    node.actions_done.add(vList)
    if vList != -1:
        node.untried_actions.remove(vList)
        # vList = get_random_actions(node.graphs, returnBest)
        # tv = tuple(vList)
        # while tv in node.actions_done:
        #     vList = get_random_actions(node.graphs, returnBest)
        # print(f"{len(node) = }, {vList = }")
        hList, cliffCount, scalar, changed = cutGraphsMCTS(node.graphs, node.cliffCount, node.scalar, vList)
    else:
        # BSS action was chosen
        hList = node.graphs
        changed = False
        cliffCount = node.cliffCount
        scalar = node.scalar
    # print(f"{len(hList) = }, {cliffCount = }, {node.depth = }")
    # If no change, then fallback to BSS
    if not changed:
        gList = []
        for g in hList:
            g = g.copy()
            zx.simplify.full_reduce(g)
            gsum = zx.simulate.replace_magic_states(g, True)
            gsum.reduce_scalar() 
            gList.extend(gsum.graphs)
        
        # print(f"{len(gList) = }, {cliffCount = }")
        hList = []
        for g in gList:
            gt = g.copy(); zx.simplify.full_reduce(gt)
            if gt.scalar.is_zero: continue

            if zx.simplify.tcount(gt) > 0 :
                gList.append(g)
            else: 
                cliffCount += 1
                scalar += g.scalar.to_number()

        # print(f"{len(hList) = }, {cliffCount = }")

    child_node = Node(graphs=hList, parent=node, vertex_cut_list=vList, scalar=scalar, cliffCount=cliffCount, 
                      depth=node.depth + 1)
    node.children.append(child_node)
    
    # if len(hList) < 10:
    #     strLine = f"{cliffCount}     {len(hList)}    | "
    #     for i in range(len(hList)):
    #         gTemp = hList[i].copy()
    #         zx.simplify.full_reduce(gTemp)
    #         strLine += f"{zx.tcount(gTemp)}  "
    #         zx.draw(gTemp,scale=20,labels=True) #TEMP
    #     print(strLine)
    return child_node

def simulation(node: Node) -> float:
    # assert len(node) > 0
    # Run a random simulation from the node's state until a terminal state
    cnode = node
    # gList = node.graphs
    # cliffCount = node.cliffCount
    # scalar = node.scalar
    while not cnode.is_terminal():
        # possible_actions = all_possible_cuts(current_diagram)
        # g = random.choice(gList)
        # vList = [random.randrange(len(g.vertices())) for g in gList]
        # nnode = Node(gList, cnode, )
        # vList = cnget_random_actions(gList, returnBest)
        all_actions = tuple(cnode.get_all_actions())
        vList = random.choice(all_actions) if all_actions else -1
        changed = False
        # vList = random.choice(tuple(node.get_all_actions()))
        # vList = random.choice(tuple(node.get_best_action()))
        # v = random.randrange(len(g.vertices()))
        # print(f"{gList = }")
        gList = cnode.graphs
        cliffCount = cnode.cliffCount
        scalar = cnode.scalar
        if vList != -1:
            gList, cliffCount, scalar, changed = cutGraphsMCTS(cnode.graphs, cnode.cliffCount, cnode.scalar, vList)

        # If no change, then fallback to BSS
        if not changed:
            hList = []
            for g in gList:
                g = g.copy()
                zx.simplify.full_reduce(g)
                gsum = zx.simulate.replace_magic_states(g, True)
                gsum.reduce_scalar() 
                hList.extend(gsum.graphs)
            
            gList = []
            for g in hList:
                gt = g.copy(); zx.simplify.full_reduce(gt)
                if gt.scalar.is_zero: continue

                if zx.simplify.tcount(gt) > 0 :
                    gList.append(g)
                else: 
                    cliffCount += 1
                    scalar += g.scalar.to_number()

        cnode = Node(gList, cnode, vList, cliffCount, scalar, cnode.depth + 1)

        # gList = hList.copy()
    # return 1 / (cliffCount + 1)  # Return evaluation of final diagram state
    return np.exp(-cnode.cliffCount)  # Return evaluation of final diagram state

def backpropagation(node: Node, reward: float):
    # Backpropagate the result of the simulation up the tree
    while node is not None:
        node.visits += 1
        node.score += reward
        node = node.parent

def monte_carlo_tree_search(root: Node, iterations=1000):
    for i in range(iterations):
        # print("mcts iter", i)
        leaf = selection(root)
        # print(root.children)
        simulation_result = simulation(leaf)
        backpropagation(leaf, simulation_result)


In [54]:
# Example (feel free to change this up)...
useExample = False
NQ = 10 # No. of qubits

strCirc = "\nqreg q[" + str(NQ) + "];"
qasmLines = list()

tSandwich(qasmLines, NQ, 8)
toff(qasmLines, NQ, 4,True)
rz(qasmLines, NQ, True)
toff(qasmLines, NQ, 2,True)
rz(qasmLines, NQ, False)
toff(qasmLines, NQ, 2,True)
rz(qasmLines, NQ, False)
rz(qasmLines, NQ, False)
tSandwich(qasmLines, NQ, 2)
toff(qasmLines, NQ, 4,True)
rz(qasmLines, NQ, True)
toff(qasmLines, NQ, 2,True)

#----------

for line in qasmLines: strCirc += line
if (useExample): strCirc = "\nqreg q[6];\nt q[4];\ncx q[2], q[4];\nt q[4];\ncx q[0], q[4];\nt q[4];\ncx q[1], q[4];\nt q[4];\nh q[3];\nccz q[2], q[4], q[3];\nh q[3];\ncx q[0], q[2];\nh q[0];\nccz q[3], q[2], q[0];\nh q[0];\ncx q[2], q[3];\nrz(0.5*pi) q[2];\nh q[1];\nccz q[4], q[0], q[1];\nh q[1];\ncx q[1], q[4];\nh q[3];\nccz q[2], q[4], q[3];\nh q[3];\ncx q[4], q[2];\nh q[1];\nccz q[0], q[4], q[1];\nh q[1];\ncx q[3], q[0];\nh q[5];\nccz q[1], q[0], q[5];\nh q[5];\ncx q[4], q[1];\nt q[0];\ncx q[1], q[0];\nt q[0];\ncx q[4], q[0];\nt q[0];\nrz(0.5*pi) q[0];"
c = zx.qasm(strCirc)
g = c.to_graph()

g.apply_state("+"*NQ)  #TEMP
g.apply_effect("+"*NQ) #TEMP

#g.normalize()
zx.draw(g, labels=True, scale=20)
print("T-count = ", zx.tcount(g))

T-count =  109


In [55]:
qubit_amount = 6
gate_count = 100
#Generate random circuit of Clifford gates
g = zx.generate.cliffordT(qubit_amount, gate_count, p_cnot=0.5, p_t=0.4)

print("Plug with some states and effects...")
g.apply_state("+"*qubit_amount)
g.apply_effect("+"*qubit_amount)
zx.draw(g, labels=True)
g = partialSimp(g)
g = g.copy()

h = g.copy()
#If running in Jupyter, draw the circuit
#Use one of the built-in rewriting strategies to simplify the circuit
zx.draw(h, labels=True)
zx.simplify.full_reduce(h)
#See the result
# zx.draw(h, labels=True)


# zx.draw(g, scale=20, labels=True)
print("T-count = ", zx.tcount(h))
gT = zx.tcount(h)

gDecomp = zx.simulate.find_stabilizer_decomp(h)
print("\nExact:   ",len(gDecomp))
print("Estimate:",7**(gT/6))

assert gT > 2, "T-count is <= 2, reroll"

Plug with some states and effects...


T-count =  14

Exact:    34
Estimate: 93.7336279558471


In [56]:
gList = [g]
root = Node(graphs = gList)
monte_carlo_tree_search(root, 50)


In [57]:
node = root
vcuts = []

while node.children:
    node = node.best_child(c_param=0)
    nvlist = node.vertex_cut_list
    if nvlist is None:
        vcuts.append(-1)
    else:
        vcuts.append(nvlist)

n = 0
if len(node):
    for g in node.graphs:
        gbss = zx.simulate.find_stabilizer_decomp(g)
        n += len(gbss)



print(f"{node.cliffCount+n = }, {len(node) = }, {node.vertex_cut_list = }, {node.depth = }, {vcuts}")
# g = circuit.copy()
# g = gList[0].copy()
cliffCount, scalar, cutList = find_decomp_ori(g)
print(f"Original {cliffCount = }, {cutList = }")

print(f"BSS: {len(gDecomp)}")
# print(node.scalar)
# print(node.scalar / (math.sqrt(node.scalar.real**2 + node.scalar.imag**2)))
# gDecomp_scalar = sum(g.scalar.to_number() for g in gDecomp)

# print(gDecomp_scalar )
# print(gDecomp_scalar / (math.sqrt(gDecomp_scalar.real**2 + gDecomp_scalar.imag**2)) )
# print(scalar)
# print(scalar / (math.sqrt(scalar.real**2 + scalar.imag**2)))

node.cliffCount+n = 224, len(node) = 8, node.vertex_cut_list = (47, 47, 18, 18), node.depth = 3, [(37,), (43, 43), (47, 47, 18, 18)]
Original cliffCount = 5, cutList = [[20], [37], -1]
BSS: 34


In [58]:

h = gList[0].copy()
if vcuts[0] != -1 and cutList[0] != -1: 
    vBest,tier,vweights_max,vweights,vtiers, possV = compWeightsM(h,False)
    # vBest,tier,vweights_max,vweights,vtiers = compWeights(h,False)
    mctsCut = vcuts[0][0]
    oriCut = cutList[0][0]
    print(f"{mctsCut = }, {oriCut = }")
    print(f"{vweights_max[mctsCut] = }, {vweights_max[oriCut] = }")
    # print(f"{vweights[mctsCut] = }, {vweights[oriCut] = }")
    print(f"{vtiers[mctsCut] = }, {vtiers[oriCut] = }")
    print(tuple(enumerate(vtiers)))


mctsCut = 37, oriCut = 20
vweights_max[mctsCut] = 2.0, vweights_max[oriCut] = 0
vtiers[mctsCut] = 1, vtiers[oriCut] = 0
((0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0), (9, 0), (10, 0), (11, 0), (12, 0), (13, 0), (14, 0), (15, 0), (16, 0), (17, 0), (18, 1), (19, 0), (20, 0), (21, 1), (22, 0), (23, 0), (24, 0), (25, 0), (26, 0), (27, 0), (28, 0), (29, 0), (30, 0), (31, 0), (32, 0), (33, 0), (34, 0), (35, 0), (36, 0), (37, 1), (38, 0), (39, 0), (40, 0), (41, 0), (42, 0), (43, 0), (44, 0), (45, 0), (46, 0), (47, 0), (48, 0), (49, 0), (50, 0), (51, 0), (52, 0), (53, 0), (54, 0), (55, 0), (56, 0), (57, 0), (58, 0), (59, 1), (60, 0), (61, 0), (62, 0), (63, 0), (64, 0), (65, 0))


In [59]:
def benchGraph(g: GraphS):      
    g = partialSimp(g)
    g = g.copy()

    h = g.copy()  
    zx.simplify.full_reduce(h)
    print("T-count = ", zx.tcount(h))
    gT = zx.tcount(h)

    gDecomp = zx.simulate.find_stabilizer_decomp(h)
    # print("\nExact:   ",len(gDecomp))
    # print("Estimate:",7**(gT/6))

    assert gT > 2, "T-count is <= 2, reroll"

    gList = [g]
    root = Node(graphs = gList)
    monte_carlo_tree_search(root, 50)

    node = root
    vcuts = []

    while node.children:
        node = node.best_child(c_param=0)
        nvlist = node.vertex_cut_list
        if nvlist is None:
            vcuts.append(-1)
        else:
            vcuts.append(nvlist)

    n = 0
    if len(node):
        for g in node.graphs:
            gbss = zx.simulate.find_stabilizer_decomp(g)
            n += len(gbss)

    # print(f"{node.cliffCount+n = }, {len(node) = }, {node.vertex_cut_list = }, {node.depth = }, {vcuts}")
    # g = circuit.copy()
    # g = gList[0].copy()
    cliffCount, scalar, cutList = find_decomp_ori(g)

    return (node.cliffCount+n, vcuts), (cliffCount, cutList), len(gDecomp)

In [67]:
def genPseudoG(qubit_amount: int = 8) -> GraphS:
    NQ = qubit_amount # No. of qubits

    strCirc = "\nqreg q[" + str(NQ) + "];"
    qasmLines = list()

    tSandwich(qasmLines, NQ, 4)
    toff(qasmLines,NQ, 2,True)
    rz(qasmLines,NQ, True)
    toff(qasmLines,NQ, 1,True)
    rz(qasmLines,NQ, False)
    toff(qasmLines,NQ, 1,True)
    rz(qasmLines,NQ, False)
    rz(qasmLines,NQ, False)    

    tSandwich(qasmLines, NQ, 4)
    toff(qasmLines,NQ, 2,True)
    rz(qasmLines,NQ, True)
    toff(qasmLines,NQ, 1,True)
    rz(qasmLines,NQ, False)
    toff(qasmLines,NQ, 1,True)
    rz(qasmLines,NQ, False)
    rz(qasmLines,NQ, False)   

    #----------

    for line in qasmLines: strCirc += line
    # if (useExample): strCirc = "\nqreg q[6];\nt q[4];\ncx q[2], q[4];\nt q[4];\ncx q[0], q[4];\nt q[4];\ncx q[1], q[4];\nt q[4];\nh q[3];\nccz q[2], q[4], q[3];\nh q[3];\ncx q[0], q[2];\nh q[0];\nccz q[3], q[2], q[0];\nh q[0];\ncx q[2], q[3];\nrz(0.5*pi) q[2];\nh q[1];\nccz q[4], q[0], q[1];\nh q[1];\ncx q[1], q[4];\nh q[3];\nccz q[2], q[4], q[3];\nh q[3];\ncx q[4], q[2];\nh q[1];\nccz q[0], q[4], q[1];\nh q[1];\ncx q[3], q[0];\nh q[5];\nccz q[1], q[0], q[5];\nh q[5];\ncx q[4], q[1];\nt q[0];\ncx q[1], q[0];\nt q[0];\ncx q[4], q[0];\nt q[0];\nrz(0.5*pi) q[0];"
    c = zx.qasm(strCirc)
    g = c.to_graph()

    # g.apply_state("+"*NQ)  #TEMP
    # g.apply_effect("+"*NQ) #TEMP

    #g.normalize()
    # zx.draw(g, labels=True, scale=20)
    # print("T-count = ", zx.tcount(g))
    return g

In [68]:
def compareAlgos(qubit_amount: int = 6, gate_count: int = 50, p_cnot: float = 0.5, p_t: float = 0.4, isPseudo=False):
    #Generate random circuit of Clifford gates
    if isPseudo:
        g = genPseudoG(qubit_amount)
    else:
        g = zx.generate.cliffordT(qubit_amount, gate_count, p_cnot=p_cnot, p_t=p_t)

    # print("Plug with some states and effects...")
    g.apply_state("+"*qubit_amount)
    g.apply_effect("+"*qubit_amount)
    # zx.draw(g, labels=True)
    # g = partialSimp(g)
    # g = g.copy()

    # h = g.copy()
    # #If running in Jupyter, draw the circuit
    # #Use one of the built-in rewriting strategies to simplify the circuit
    # # zx.draw(h, labels=True)
    # zx.simplify.full_reduce(h)
    # print("T-count = ", zx.tcount(h))
    # gT = zx.tcount(h)

    # gDecomp = zx.simulate.find_stabilizer_decomp(h)
    # # print("\nExact:   ",len(gDecomp))
    # # print("Estimate:",7**(gT/6))

    # assert gT > 2, "T-count is <= 2, reroll"

    # gList = [g]
    # root = Node(graphs = gList)
    # monte_carlo_tree_search(root, 50)

    # node = root
    # vcuts = []

    # while node.children:
    #     node = node.best_child(c_param=0)
    #     nvlist = node.vertex_cut_list
    #     if nvlist is None:
    #         vcuts.append(-1)
    #     else:
    #         vcuts.append(nvlist)

    # n = 0
    # if len(node):
    #     for g in node.graphs:
    #         gbss = zx.simulate.find_stabilizer_decomp(g)
    #         n += len(gbss)

    # # print(f"{node.cliffCount+n = }, {len(node) = }, {node.vertex_cut_list = }, {node.depth = }, {vcuts}")
    # # g = circuit.copy()
    # # g = gList[0].copy()
    # cliffCount, scalar, cutList = find_decomp_ori(g)
    # print(f"Original {cliffCount = }, {cutList = }")

    # print(f"BSS: {len(gDecomp)}")

    # return (node.cliffCount+n, vcuts), (cliffCount, cutList), len(gDecomp)
    return benchGraph(g)

In [69]:
def repeatCompare(n: int = 20, qubit_amount: int = 6, gate_count: int = 50, p_cnot: float = 0.5, p_t: float = 0.4):
    bss_wins = 0
    m_wins = 0
    o_wins = 0

    for i in range(n):
        print("iter", i)
        try:
            if i < n/2:
                    mres, ores, bres = compareAlgos(qubit_amount, gate_count, p_cnot, p_t)
            else:
                print("isPseudo")
                mres, ores, bres = compareAlgos(isPseudo=True)
        except AssertionError:
            continue
            
        
        mcnt, mcuts = mres
        ocnt, ocuts = ores

        if bres <= mcnt and bres <= ocnt:
            bss_wins += 1
        elif ocnt <= mcnt:
            o_wins += 1
        else:
            m_wins += 1

        print(f"{bres = }, {mcnt = }, {ocnt = }, {mcuts = }, {ocuts = }")
    return bss_wins, m_wins, o_wins



In [72]:
bss_wins, m_wins, o_wins = repeatCompare(4, qubit_amount=6, gate_count=50, p_cnot=0.4, p_t=0.4)
print(f"{bss_wins = }, {m_wins = }, {o_wins = }")

iter 0
T-count =  9
bres = 8, mcnt = 4, ocnt = 4, mcuts = [(25,), -1], ocuts = [[25], -1, -1, -1]
iter 1
T-count =  6
bres = 6, mcnt = 2, ocnt = 3, mcuts = [(26,)], ocuts = [[4], [22]]
iter 2
isPseudo
T-count =  16
bres = 54, mcnt = 4, ocnt = 8, mcuts = [(4,), (40, 42)], ocuts = [[3], [3, 3], [2, 2, 2, 2]]
iter 3
isPseudo
T-count =  23
bres = 368, mcnt = 3, ocnt = 9, mcuts = [(3,), (0,)], ocuts = [[1], [2, 2], [0, 0, 0, 0], [10]]
bss_wins = 0, m_wins = 3, o_wins = 1


In [ ]:
benchGraph(pseudog)

In [ ]:

print(f"{root.get_best_action() = }")
gl, cl, sc, changed, ccs = cutGraphs(gList,cliffCount,scalar,False)
gl2, cl2, sc2, changed2, ccs2 = cutGraphs(gl,cl,sc,False)
print(f"{gl = }, {cl = }, {changed = }, {ccs = }")
print(f"{gl2 = }, {cl2 = }, {changed2 = }, {ccs2 = }")

root.get_best_action() = (2,)
gl = [Graph(63 vertices, 90 edges), Graph(63 vertices, 90 edges)], cl = 2, changed = True, ccs = [2]
gl2 = [Graph(54 vertices, 79 edges), Graph(54 vertices, 79 edges), Graph(54 vertices, 79 edges), Graph(54 vertices, 79 edges)], cl2 = 2, changed2 = True, ccs2 = [6, 6]


In [ ]:
if isinstance(vcuts[0], tuple): 
    cutChoice = vcuts[0][0]
    h = gList[0].copy()
    h0, h1 = apply_cut(h, cutChoice)

    zx.draw(h1, labels=True)
    for g, s in zx.simplify.full_reduce_iter(h1):
        # zx.draw(g,labels=True)
        print(s)


to_gh
id1
spider1
pivot1
pivot2
lcomp1
lcomp2
lcomp3
pivot_gadget1
pivot_gadget2
pivot_gadget3
pivot_gadget4
clifford -> to_gh
clifford -> id1
clifford -> spider1
clifford -> lcomp1
clifford -> lcomp2
gadget -> gadget1
interior_clifford -> to_gh
interior_clifford -> id1
interior_clifford -> spider1
interior_clifford -> pivot1
clifford -> to_gh
interior_clifford -> to_gh


In [ ]:
h = gList[0].copy()
hList = []

zx.draw(h, labels=True)

zx.simplify.full_reduce(h)
gsum = zx.simulate.replace_magic_states(h, True)
gsum.reduce_scalar() 
hList.extend(gsum.graphs)
print(f"{len(hList) = }")

for g in hList:
    zx.draw(g, labels=True)
    print(f"{zx.tcount(g) = }, {g.scalar = }")

len(hList) = 7


zx.tcount(g) = 7, g.scalar = Scalar(-0.15+0.15i = -3.41+3.41isqrt(2)^-9)


zx.tcount(g) = 7, g.scalar = Scalar(-0.03+0.03i = 0.59-0.59iexp(1ipi)sqrt(2)^-9)


zx.tcount(g) = 7, g.scalar = Scalar(-0.04-0.04i = -0.35+0.35iexp(1/2ipi)sqrt(2)^-6)


zx.tcount(g) = 7, g.scalar = Scalar(-0.06-0.06i = -0.50+0.50iexp(1/2ipi)sqrt(2)^-6)


zx.tcount(g) = 0, g.scalar = Scalar(0.06-0.00i = 0.25-0.25iexp(1/4ipi)sqrt(2)^-5)


zx.tcount(g) = 7, g.scalar = Scalar(0.06-0.06i = -0.35+0.35iexp(1ipi)sqrt(2)^-5)


zx.tcount(g) = 7, g.scalar = Scalar(0.18-0.00i = -0.35+0.35iexp(5/4ipi)sqrt(2)^-3)


In [ ]:
h = gList[0].copy()
zx.draw(h, labels=True)
a = h.to_tikz()
a

# print(a)
# h.to_tikz()
# h.to_matrix()


'\n\\begin{tikzpicture}\n    \\begin{pgfonlayer}{nodelayer}\n        \\node [style=Z dot] (0) at (0.00, 0.00) {};\n        \\node [style=Z phase dot] (1) at (0.00, -2.00) {$\\frac{\\pi}{4}$};\n        \\node [style=Z phase dot] (2) at (0.00, -3.00) {$\\frac{3\\pi}{4}$};\n        \\node [style=Z phase dot] (3) at (0.00, -4.00) {$\\pi$};\n        \\node [style=Z phase dot] (4) at (0.00, -5.00) {$\\frac{\\pi}{4}$};\n        \\node [style=X dot] (5) at (2.00, 0.00) {};\n        \\node [style=Z phase dot] (6) at (5.00, 0.00) {$\\frac{3\\pi}{4}$};\n        \\node [style=X dot] (7) at (6.00, -5.00) {};\n        \\node [style=Z phase dot] (8) at (14.00, -5.00) {$\\frac{3\\pi}{4}$};\n        \\node [style=Z phase dot] (9) at (15.00, -1.00) {$\\frac{\\pi}{2}$};\n        \\node [style=X dot] (10) at (18.00, -2.00) {};\n        \\node [style=Z dot] (11) at (21.00, -2.00) {};\n        \\node [style=X dot] (12) at (21.00, 0.00) {};\n        \\node [style=X dot] (13) at (22.00, -1.00) {};\n        \\

In [ ]:
# a = '\n\\begin{tikzpicture}\n    \\begin{pgfonlayer}{nodelayer}\n        \\node [style=Z phase dot] (0) at (0.00, 0.00) {$\\frac{\\pi}{4}$};\n        \\node [style=Z phase dot] (1) at (0.00, -1.00) {$\\frac{3\\pi}{4}$};\n        \\node [style=Z dot] (2) at (0.00, -2.00) {};\n        \\node [style=X dot] (3) at (3.00, -2.00) {};\n        \\node [style=Z phase dot] (4) at (5.00, -2.00) {$\\frac{\\pi}{4}$};\n        \\node [style=X dot] (5) at (7.00, -1.00) {};\n        \\node [style=Z phase dot] (6) at (9.00, -1.00) {$\\frac{\\pi}{4}$};\n        \\node [style=X dot] (7) at (10.00, -1.00) {};\n        \\node [style=X dot] (8) at (11.00, 0.00) {};\n        \\node [style=Z phase dot] (9) at (12.00, 0.00) {$\\frac{\\pi}{4}$};\n        \\node [style=X dot] (10) at (13.00, -2.00) {};\n        \\node [style=Z dot] (11) at (14.00, -2.00) {};\n        \\node [style=X dot] (12) at (14.00, 0.00) {};\n        \\node [style=Z phase dot] (13) at (15.00, 0.00) {$\\frac{\\pi}{4}$};\n        \\node [style=X phase dot] (14) at (16.00, -2.00) {$\\frac{\\pi}{2}$};\n        \\node [style=Z dot] (15) at (17.00, -2.00) {};\n        \\node [style=Z phase dot] (16) at (19.00, 0.00) {$\\frac{\\pi}{4}$};\n        \\node [style=X dot] (17) at (19.00, -2.00) {};\n        \\node [style=Z phase dot] (18) at (20.00, -2.00) {$\\frac{\\pi}{4}$};\n    \\end{pgfonlayer}\n    \\begin{pgfonlayer}{edgelayer}\n        \\draw (0) to (5);\n        \\draw (0) to (7);\n        \\draw (0) to (8);\n        \\draw (1) to (3);\n        \\draw (1) to (5);\n        \\draw (2) to (3);\n        \\draw (3) to (4);\n        \\draw (4) to (8);\n        \\draw (4) to (10);\n        \\draw (4) to (5);\n        \\draw (5) to (6);\n        \\draw (6) to (7);\n        \\draw (7) to (15);\n        \\draw (7) to (13);\n        \\draw (7) to (16);\n        \\draw (8) to (9);\n        \\draw (9) to (10);\n        \\draw (9) to (12);\n        \\draw (10) to (11);\n        \\draw (11) to (12);\n        \\draw (11) to (14);\n        \\draw (12) to (13);\n        \\draw (14) to (15);\n        \\draw (15) to (17);\n        \\draw (16) to (17);\n        \\draw (17) to (18);\n    \\end{pgfonlayer}\n\\end{tikzpicture}\n'
# # b = GraphS.from_tikz(a)
# b = c.copy()
# # c = b.subgraph_from_vertices()
# zx.draw(b, labels=True)
# # print(a)
# bm0, bm1 = apply_cut(b, 4)
# bp0, bp1 = apply_cut(b, 9)



In [ ]:
steps_bm0 = []
for g, s in zx.simplify.full_reduce_iter(bm0.copy()):
    # zx.draw(g,labels=True)
    print(s)
    steps_bm0.append(g.to_tikz())

NameError: name 'bm0' is not defined

In [ ]:
steps_bm1 = []
for g, s in zx.simplify.full_reduce_iter(bm1.copy()):
    # zx.draw(g,labels=True)
    print(s)
    steps_bm1.append(g.to_tikz())

to_gh
pivot1
lcomp1
pivot_gadget1
pivot_gadget2
clifford -> to_gh
clifford -> id1
clifford -> spider1
clifford -> lcomp1
clifford -> lcomp2
gadget -> gadget1
interior_clifford -> to_gh
interior_clifford -> id1
interior_clifford -> spider1
clifford -> to_gh
interior_clifford -> to_gh


In [ ]:
print(len(steps_bm0),len(steps_bm1) )

13 16


In [ ]:
# for i, step in enumerate(steps_bm0):
#     with open(f"{i}_bm0.tikz", 'w') as f:
#        f.write(step)

In [ ]:

# c = b.copy()
# c.set_row(3, 1)
# c.set_row(4, 2)
# c.set_row(5, 2)
# c.set_row(6, 3)
# c.set_row(7, 4)
c.set_row(8, 3)
# c.set_row(9, 5)
# c.set_row(10, 5)
# c.set_row(11, 6)
# c.set_row(12, 6)
# c.set_row(13, 7)
# c.set_row(14, 7)
# c.set_row(15, 8)
# c.set_row(16, 9)
# c.set_row(17, 9)
# c.set_row(18, 10)

zx.draw(c, labels=True)
for v in c.vertices():
    print(v, c.row(v))

0 0.0
1 0.0
2 0.0
3 1
4 2
5 2
6 3
7 4
8 3
9 5
10 5
11 6
12 6
13 7
14 7
15 8
16 9
17 9
18 10


In [ ]:
# with open(f"diagram.tikz", 'w') as f:
#     f.write(c.to_tikz())


In [ ]:
bmbm = bm0.copy()
bmbm = partialSimp(bmbm)

bmbm.set_row(2, 1)
bmbm.set_row(3, 2)
bmbm.set_row(4, 3)
bmbm.set_row(5, 2)
bmbm.set_row(6, 3)
bmbm.set_row(7, 1)
bmbm.set_row(8, 3)
bmbm.set_row(9, 4)
bmbm.set_row(10, 4)
bmbm.set_row(11, 5)




zx.draw(bmbm, labels=True)
for v in bmbm.vertices():
    print(v, bmbm.row(v))

0 0.0
1 0.0
2 1
3 2
4 3
5 2
6 3
7 1
8 3
9 4
10 4
11 5


In [ ]:
steps_bm0 = []
steps_bp0 = []
for g, s in zx.simplify.full_reduce_iter(bmbm.copy()):
    # zx.draw(g,labels=True)
    print(s)
    steps_bm0.append(g.to_tikz())

to_gh
pivot1
lcomp1
lcomp2
lcomp3
pivot_gadget1
pivot_gadget2
clifford -> to_gh
clifford -> id1
clifford -> spider1
clifford -> lcomp1
interior_clifford -> to_gh


In [ ]:
# for i, step in enumerate(steps_bm0):
#     with open(f"{i}_bm0.tikz", 'w') as f:
#        f.write(step)

In [ ]:
# with open(f"diagram_part.tikz", 'w') as f:
#     f.write(bmbm.to_tikz())

In [ ]:
bpbp0 = bp0.copy()
bpbp1 = bp1.copy()

bpbp0 = partialSimp(bpbp0)

bpbp0.set_row(4, 1)
bpbp0.set_row(5, 2)
bpbp0.set_row(6, 3)
bpbp0.set_row(7, 4)
bpbp0.set_row(8, 2)
bpbp0.set_row(9, 5)
bpbp0.set_row(10, 6)

zx.draw(bpbp0, labels=True)
for v in bpbp0.vertices():
    print(v, bpbp0.row(v))

# zx.simplify.full_reduce(bpbp0)
# zx.simplify.full_reduce(bpbp1)
# print(zx.tcount(bpbp0))
# print(zx.tcount(bpbp1))

0 0.0
1 0.0
2 0.0
3 1
4 1
5 2
6 3
7 4
8 2
9 5
10 6


In [ ]:
# steps_bm0 = []
steps_bp0 = []
for g, s in zx.simplify.full_reduce_iter(bpbp0.copy()):
    # zx.draw(g,labels=True)
    print(s)
    steps_bp0.append(g.to_tikz())

spider1
to_gh
pivot1
pivot2
lcomp1
clifford -> to_gh
interior_clifford -> to_gh


In [ ]:
# for i, step in enumerate(steps_bp0):
#     with open(f"{i}_bp0.tikz", 'w') as f:
#        f.write(step)

In [ ]:
# with open(f"diagram_old.tikz", 'w') as f:
#     f.write(bpbp0.to_tikz())

```Python
node.cliffCount+n = 2, len(node) = 0, node.vertex_cut_list = (4,), node.depth = 1, [(4,)]
data = [[Graph(18 vertices, 23 edges), Graph(18 vertices, 23 edges)], 0, 0j, True, [9]]
data = [[Graph(11 vertices, 12 edges), Graph(10 vertices, 11 edges)], 0, 0j, False, [-1, -1]]
len(hList) = 3, cliffCount = 0
len(gList) = 0, cliffCount = 3
treeDepth = 2
Original cliffCount = 3, cutList = [[9], -1]
BSS: 2

mctsCut = 4, oriCut = 9
vweights_max[mctsCut] = 3.0, vweights_max[oriCut] = 3.0
vtiers[mctsCut] = 1, vtiers[oriCut] = 2
((0, 1), (1, 0), (2, 0), (3, 0), (4, 1), (5, 0), (6, 0), (7, 0), (8, 0), (9, 2), (10, 0), (11, 1), (12, 0), (13, 0), (14, 0), (15, 0), (16, 0), (17, 0), (18, 0))
```